# eu_and_v3 — single-switch and AND reports from sampled stem opening

One run builds two reports (spec: `eu_and_v3_spec.md`):

* **Part 1, single-switch report.** For each of the first `N_DESIGNS` designs of the NucSyn CSV, **Open** is the
  expected length of the **longest run of consecutive** open `stem_up` nt (open = not paired to a `stem_down` nt),
  estimated by sampling structures from the folding ensemble; **E[k]**, the expected number of open nt anywhere on
  the stem, is listed next to it. **ON** = Open(switch + trigger) and **OFF** = the worst of Open(switch alone) and
  `N_CONTROLS` mismatched-trigger controls, both as % of the stem; designs are ranked by **ON - OFF** (percentage
  points). Designs within the seed-to-seed noise of the best form a tied group, ordered by dG. The sampler is checked
  against the exact base-pair-probability matrix. Every run writes into its own folder `var/reports/eu_and_v3_run_<time>/`. Part 1 writes `eu_and_v3_report_<time>.pdf` (a PDF: the table and about two pages of details per design) and a CSV of all designs.
* **Part 2, AND report.** The top `AND_TOP_N` designs of part 1 are fused in all ordered pairs (A = 5' half, B = 3' half,
  with and without a spacer; pairs with too-similar triggers and constructs with an AUG before the second hairpin's
  `stem_down` are excluded); each candidate is folded alone, with trigger A, with trigger B and with both, and with
  mismatched triggers. **ON** = E[min(r_A / L_A, r_B / L_B)] in the +A+B state (each hairpin's longest open run as a fraction of its own stem,
  the smaller of the two, in the same structure); **OFF** = the worst over the OFF states and controls of the same quantity. Candidates are ranked by **ON - OFF** of this
  joint quantity (the sum-based ON, E[r_A + r_B] / (L_A + L_B), is listed for information only). Writes `eu_and_v3_and_report_<time>.pdf` (about five pages of
  details per candidate), a CSV of all candidates and the data for an interactive page.

The definitions are written out in the reports themselves. All parameters of both parts are in the one parameter cell.


## Setup

In [ ]:
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_fixtures.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break
for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_switch_construct.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Parameters — every tunable in one place

In [ ]:
# ---- data source ------------------------------------------------------------
CSV_PATH = (
    "/Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/"
    "output_T3/targets/sele/basic_switch/results_ranked.csv"
)
COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
    "trigger_start": "trigger_start_pos",
    "trigger_end": "trigger_end_pos",
}
CARRY_METRICS = ("sim_score", "mcc", "switch_toehold_unpaired_prob_mean", "energy_trigger_mfe")
N_DESIGNS = 150            # first rows of the CSV that are scored

# ---- folding and sampling ------------------------------------------------------
TEMPERATURE_C = 37.0
SALT_MOLAR = 0.1            # monovalent salt, mol/L (ViennaRNA salt correction; NUPACK sodium); no magnesium
N_SAMPLES = 2000          # structures drawn per state
SEED = 1                  # ViennaRNA's generator is process-global; fixed so a rerun repeats
MAX_WORKERS = 6           # process-pool width
CHECK_SEED_OFFSET = 1000  # a second, independent sampling run (seed + this) only measures the noise

# ---- verification --------------------------------------------------------------
VERIFY_SIGMA = 4.0        # allowed disagreement with the matrix, in standard errors of the sampled mean
VERIFY_FLOOR_NT = 0.02    # plus this many nt (rounding of the matrix, rare draws)

# ---- controls (mismatched triggers) --------------------------------------------
N_CONTROLS = 5            # mismatched triggers per design; OFF is the worst of the alone state and these
CONTROL_MAX_OVERLAP_NT = 8  # a mismatched trigger shares at most this many nt of identity with the real one
CONTROL_EXTRA_TRIGGERS = ()  # optional extra RNA sequences offered as mismatched controls besides the other designs' triggers

# ---- report (both parts) --------------------------------------------------------
OFF_FLOOR_PCT = 1.0       # OFF (% of the stem) is not allowed below this in the ratio ON / OFF
REPORT_TOP_N = 20         # rows listed in each report table (the CSV exports have all)
OUTPUT_DIR = None         # base folder (None = var/reports of the repository); every run writes into its own timestamped subfolder
MAKE_PDF = True           # False keeps the HTML source of each PDF instead (no browser needed; for tests)

# Re-render a part from a saved results file instead of sampling again (None = sample).
# The engine named in the reports (a label only; this notebook folds with ViennaRNA through FoldEngine).
ENGINE_NAME = "ViennaRNA"
# True: sample the control states like the others (only for an engine whose sampler is sound on them);
# False: take each control from the exact matrix as an upper bound of its Open (ViennaRNA's sampler is
# unreliable on mismatched-trigger states).
CONTROLS_SAMPLED = True
# Who produced the sampled values of a saved results file (shown in both reports); None = this notebook's own sampler.
SAMPLER_NOTE = None
RESULTS_TO_LOAD = None       # part 1: <run folder>/eu_and_v3_results_<time>.json
RESULTS_TO_LOAD_AND = None   # part 2: <run folder>/eu_and_v3_and_results_<time>.json

# ---- part 2: the AND report -----------------------------------------------------
AND_TOP_N = 20            # the top designs of part 1 (tied group first, then ON - OFF) fused in all ordered pairs

# AND construct
CUT_5P_BEFORE = "kozak"   # the 5' half keeps everything before its OWN kozak
KEEP_3P_AFTER = "prefix"  # the 3' half keeps everything after its OWN prefix
SPACER_VARIANTS = ("no_spacer", "spacer")   # both are run and reported separately
SPACER_MIN_LENGTH = 15    # the spacer pool: random A/U-biased sequences, lengths cycle min..max (nt)
SPACER_MAX_LENGTH = 20
SPACER_SEED = 1           # the pool is generated once from this seed
SPACER_CANDIDATE_POOL_SIZE = 150   # per pair the candidate least complementary to toeholds and triggers is used
FAKE_MAX_SEQ_OVERLAP_NT = 8   # trigger A vs trigger B: longest identical / complementary run allowed
WOBBLE_MAX_FRACTION = 0.50    # max share of a counted complementary run that may be G.U wobble
EXCLUDE_UPSTREAM_AUG = True   # drop constructs with an AUG before the 2nd hairpin's stem_down (a competing start codon)
AND_MAX_CANDIDATES = None     # sample at most this many AND candidates (None = all); for quick trials

# Data for an interactive page (filters + expandable 2D structures and heat maps): rows of ALL candidates
# and the detail panels of the best ARTIFACT_DETAIL_N, written to <run folder>/eu_and_v3_and_artifact_<time>/.
EXPORT_ARTIFACT = True
ARTIFACT_DETAIL_N = 300


# Part 1 — single-switch report

## Load the designs and sample both states

In [ ]:
import datetime, html, json, math, statistics, subprocess, time
from pathlib import Path

from _joint_state_parallel import pick_fake_triggers, use_fold_engine
from _report_stats import median_text, pct_text, run_provenance, state_ok
from _report_figures import construct_figure_svg
from _sampled_stem_parallel import DesignJob, run_parallel_designs
from _switch_construct import load_ranked_designs

folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C, salt_molar=SALT_MOLAR)
use_fold_engine(folder)   # the pool forks: publish the one engine first

_report_root = Path.cwd()
if OUTPUT_DIR is not None:
    _report_root = Path(OUTPUT_DIR)
else:
    for _base in (_report_root, *_report_root.parents):
        if (_base / "pyproject.toml").exists():
            _report_root = _base / "var" / "reports"
            break
# every run gets its own folder, named by the start time, so runs never mix or overwrite each other
RUN_DIR_NAME = f"eu_and_v3_run_{datetime.datetime.now():%Y%m%dT%H%M%S}"
_report_root = _report_root / RUN_DIR_NAME
_report_root.mkdir(parents=True, exist_ok=True)
print(f"this run writes to {_report_root}")

designs, problems = load_ranked_designs(
    CSV_PATH, folder, columns=COLUMNS, carry_metrics=CARRY_METRICS, limit=N_DESIGNS,
)
for _line_no, _reason in problems[:10]:
    print(f"  skipped line {_line_no}: {_reason}")
print(f"loaded {len(designs)} designs ({len(problems)} rows skipped)")

# what a reader needs to verify a report: code revision, input file checksum, folding model
RUN_INFO = run_provenance(CSV_PATH, folder.provenance(), Path.cwd())
_PROVENANCE_ROWS = [
    ("code_revision", html.escape(RUN_INFO["code_revision"]), "git commit of the code that rendered this report"),
    ("input_file", html.escape(RUN_INFO["input_file"]), "the CSV the designs were read from"),
    ("folding_model", html.escape(RUN_INFO["folding_model"]), "model settings of every fold in this report"),
]
if RESULTS_TO_LOAD is not None or RESULTS_TO_LOAD_AND is not None:
    _PROVENANCE_ROWS.append((
        "saved_results", html.escape(", ".join(Path(x).name for x in (RESULTS_TO_LOAD, RESULTS_TO_LOAD_AND) if x)),
        "sampling was not repeated: the sampled values come from these saved files"))
if SAMPLER_NOTE:
    _PROVENANCE_ROWS.append(("sampler", html.escape(SAMPLER_NOTE), "who produced the sampled values of this report"))
_SAMPLER_BANNER = f" <b>Sampled values:</b> {html.escape(SAMPLER_NOTE)}." if SAMPLER_NOTE else ""
# ---- wording that depends on the engine and on how the controls are computed (placeholders of both reports)
_VIENNA = ENGINE_NAME == "ViennaRNA"
_E = ENGINE_NAME
_WORDS = {
    "FIG_CONSTRUCT": construct_figure_svg(),
    "SALT": f"{SALT_MOLAR:g}",
    "ENGINE": f"{ENGINE_NAME} {folder.versions().get(ENGINE_NAME, '')}".strip(),
    "ENGINE_NAME": ENGINE_NAME,
    "SAMPLER_METHOD": ("ViennaRNA stochastic backtracking, <code>FoldEngine.sample_structures</code>" if _VIENNA
                       else f"the sampler of {ENGINE_NAME} for the equilibrium ensemble"),
}
_HOW = (
    r"<p><b>How the sampler draws a structure.</b> @@WHO@@ first computes, for every segment of the strands, the summed Boltzmann weight "
    r"\(e^{-E(s)/RT}\) of all structures that segment can form (the same dynamic programme that gives the partition function and the "
    r"base-pair probabilities; the strands are taken in the order written in the state). It then <i>backtracks</i> from the whole complex at random: "
    r"at each step it chooses between the possibilities for the next nucleotide (left unpaired, paired with one of the possible partners, or "
    r"the segment split into two independent parts), each with probability proportional to the weight stored for that choice, and continues into the "
    r"pieces until every nucleotide is assigned. One walk gives one complete structure, and the probability of obtaining a particular structure "
    r"\(s\) is exactly its equilibrium probability \(P(s)\), so the draws need no reweighting. The tables are filled once and every further draw "
    r"costs only a backtrack, which is why thousands of draws per state are affordable. @@EXTRA@@</p>")
_WORDS["SAMPLER_HOW"] = (
    _HOW.replace("@@WHO@@", "ViennaRNA's stochastic backtracking (<code>pbacktrack</code>)")
        .replace("@@EXTRA@@", "The draws are independent, and the sampler is seeded, so a rerun with the same seed repeats them.")
    if _VIENNA else
    _HOW.replace("@@WHO@@", "NUPACK's <code>sample</code>")
        .replace("@@EXTRA@@", "NUPACK's sampler cannot be seeded, so a rerun draws new structures; the strands of a complex form a ring, so rotations of the strand order are the same complex."))
_STRUCT_NOTE = "" if _VIENNA else " The 2D drawings use ViennaRNA's layout routine for geometry only; every energy and probability is from " + ENGINE_NAME + "."
_WORDS_SINGLE = {
    "STRAND_ORDER": (
        "<p><b>Strand order.</b> For the switch + trigger state the trigger strand is written first in the sampler (the order that matched the matrix); the switch alone is a single strand.</p>"
        if _VIENNA else
        f"<p><b>Strand order.</b> The switch + trigger state is a two-strand complex; in the tests of this project the sampler of {ENGINE_NAME} agreed with its exact matrix with either strand first.</p>"),
    "SAMPLER_LIMIT": (
        "<p><b>Limitation.</b> ViennaRNA's sampler was found to return wrong draws for some multi-strand states, depending on which strand is written first\n"
        "(structures whose energy is far above the ensemble's, always with the stem open, so Open and \\(\\mathbb{E}[k]\\) are biased upwards). The strand order is chosen to avoid this, and no draw is filtered or\n"
        "corrected. The outcome of the check for every state is in Appendix A; a state that failed it should not be trusted.</p>"
        if _VIENNA else
        f"<p><b>Limitation.</b> The sampler of {ENGINE_NAME} is checked, state by state, against its own exact base-pair probability matrix (Appendix A); nothing is filtered or corrected. "
        "The check covers the expected number of open nucleotides, not the longest runs; a state that failed it should not be trusted." + _STRUCT_NOTE + "</p>"),
}
_WORDS_AND = {
    "SAMPLER_LIMIT": (
        "<p><b>Limitation.</b> ViennaRNA's sampler can return wrong draws that depend on the order in which the strands are written. Here the trigger\n"
        "of the hairpin nearest the 5&prime; end (A) is written before the fused strand and the trigger of the 3&prime; hairpin (B) after it; this order was\n"
        "found to match the matrix, but nothing is filtered or corrected. The outcome of the check for every state is in Appendix A; a state that failed it should not be trusted (biased upwards: the wrong draws always have the stems open).</p>"
        if _VIENNA else
        f"<p><b>Limitation.</b> The sampler of {ENGINE_NAME} is checked, state by state, against its own exact base-pair probability matrix (Appendix A); nothing is filtered or corrected. "
        "The three-strand state +A+B is written as the fused strand, trigger B, trigger A, the nested order in which both triggers can be bound; the other order cannot represent that and would show no joint opening. "
        "The check covers the expected number of open nucleotides, not the longest runs; a state that failed it should not be trusted." + _STRUCT_NOTE + "</p>"),
}
if CONTROLS_SAMPLED:
    _CHECKED_NOTE_S = f" Each control state is checked against the matrix like the others (Appendix A); {_E}'s sampler can disagree with the matrix on mismatched states, and the Control is then too high."
    _WORDS_SINGLE.update({
        "CTRL_PARA": ("<p><b>Controls.</b> To see whether Open responds only to the <i>right</i> trigger, every design is also folded with @@N_CONTROLS@@ mismatched triggers. Each of these control states is sampled exactly like the others, and its Open is the expected longest run of open nucleotides. The <i>Control</i> is the largest of them, \\(100\\cdot\\max\\mathrm{Open}_{\\text{control}}/L\\), and OFF is the larger of it and the alone value, so a switch that any mismatched trigger opens is penalised. The Control column shows it; ON &minus; OFF therefore already contains it." + _CHECKED_NOTE_S + "</p>"),
        "CTRL_MED": "median Control",
        "CTRL_HDR": "Control",
        "CTRL_DEF": "",
        "CTRL_LIMIT": ", sampled with the same sampler as the other states (Appendix A shows whether it agrees with the matrix on them)",
    })
    _WORDS_AND.update({
        "CTRL_PARA": ("<p><b>Controls.</b> To see whether Open responds only to the <i>right</i> triggers, every candidate is also folded in @@N_CONTROLS@@ control states, each with two mismatched triggers in place of A and B (real triggers of other designs sharing at most @@CONTROL_OVERLAP@@ nt of identity with either real trigger). Each control state is sampled exactly like the others and its Open is the same joint quantity \\(\\mathbb{E}[\\min(r_A/L_A,r_B/L_B)]\\); the largest of the @@N_CONTROLS@@ is the <i>Control</i>." + _CHECKED_NOTE_S.replace("Each control state is checked", "Each control state is also checked") + " OFF is the largest of Open(alone), Open(+A), Open(+B) and the Control (all in %), so a construct that mismatched triggers can open is penalised.</p>"),
        "CTRL_MED": "median Control",
        "CTRL_HDR": "Control",
        "CTRL_DEF": "",
        "CTRL_LIMIT": "; the controls are sampled with the same sampler as the other states (Appendix A shows whether it agrees with the matrix on them)",
    })
else:
    _WORDS_SINGLE.update({
        "CTRL_PARA": ("<p><b>Controls.</b> To see whether Open responds only to the <i>right</i> trigger, every design is also folded with @@N_CONTROLS@@ mismatched triggers. For each, the exact \\(\\mathbb{E}[k]_{\\text{matrix}}\\) of the switch with that trigger is an <b>upper bound</b> on its Open (\\(\\mathrm{Open}\\le\\mathbb{E}[k]\\)); the matrix is used instead of sampling because " + _E + "'s sampler is unreliable for such mismatched states (its draws leave the stem open although the trigger cannot bind). The <i>Control</i> is the largest of these bounds, \\(100\\cdot\\max\\mathbb{E}[k]_{\\text{matrix}}/L\\), and OFF is the larger of it and the alone value, so a switch that any mismatched trigger opens is penalised. The Control column shows it; ON &minus; OFF therefore already contains it.</p>"),
        "CTRL_MED": "median control (worst of @@N_CONTROLS@@ mismatched triggers, matrix upper bound)",
        "CTRL_HDR": "\\(\\mathbb{E}[k]\\)<br>matrix (&ge; Open)<br>up to @@N_CONTROLS@@ mismatched triggers",
        "CTRL_DEF": "Control = the largest exact \\(\\mathbb{E}[k]\\) of the switch with any of @@N_CONTROLS@@ mismatched triggers, an upper bound of its Open.",
        "CTRL_LIMIT": "",
    })
    _WORDS_AND.update({
        "CTRL_PARA": ("<p><b>Controls.</b> To see whether Open responds only to the <i>right</i> triggers, every candidate is also folded in @@N_CONTROLS@@ control states, each with two mismatched triggers in place of A and B (real triggers of other designs sharing at most @@CONTROL_OVERLAP@@ nt of identity with either real trigger). Because the sampler is unreliable for mismatched states, each control is taken from the exact matrix as the upper bound \\(\\min(\\mathbb{E}[k_A]/L_A,\\mathbb{E}[k_B]/L_B)\\ge\\mathrm{Open}\\), and the largest of the @@N_CONTROLS@@ is the <i>Control</i>. OFF is the largest of Open(alone), Open(+A), Open(+B) and the Control (all in %), so a construct that mismatched triggers can open is penalised.</p>"),
        "CTRL_MED": "median control (worst of @@N_CONTROLS@@ mismatched states, matrix upper bound)",
        "CTRL_HDR": "\\(\\min(\\mathbb{E}[k_A]/L_A,\\mathbb{E}[k_B]/L_B)\\)<br>matrix (&ge; Open)",
        "CTRL_DEF": "The Control is the largest exact upper bound over @@N_CONTROLS@@ constructs with two mismatched triggers instead of A and B; it is part of OFF.",
        "CTRL_LIMIT": "",
    })
_CTRL_CSV = "control_open_pct" if CONTROLS_SAMPLED else "control_matrix_upper_bound_pct"
for _name, _value, _note in _PROVENANCE_ROWS:
    print(f"{_name}: {_value}")

if RESULTS_TO_LOAD is None:
    # control: for each design a mismatched trigger, i.e. the real trigger of another design that
    # shares at most CONTROL_MAX_OVERLAP_NT nt of identical sequence with the true trigger
    _control_pool = designs + [{"rank": f"extra{_k}", "trigger": _t} for _k, _t in enumerate(CONTROL_EXTRA_TRIGGERS)]
    for _i, _d in enumerate(designs):
        _fakes = pick_fake_triggers((_d["trigger"],), {_d["rank"]}, _control_pool, N_CONTROLS,
                                    start_offset=_i + 1, max_overlap_nt=CONTROL_MAX_OVERLAP_NT)
        _d["control_triggers"] = tuple(_f["trigger"] for _f in _fakes)
        _d["control_ranks"] = [_f["rank"] for _f in _fakes]
    jobs = [
        DesignJob(
            key=_d["rank"], switch=_d["switch"], trigger=_d["trigger"],
            stem_up=_d["domains"]["stem_up"], stem_down=_d["domains"]["stem_down"],
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, seed=SEED, controls=_d["control_triggers"],
            sample_controls=CONTROLS_SAMPLED,
        )
        for _d in designs
    ]
    _t0 = time.time()
    results = run_parallel_designs(jobs, max_workers=MAX_WORKERS)
    check_results = run_parallel_designs(
        [j._replace(seed=SEED + CHECK_SEED_OFFSET) for j in jobs], max_workers=MAX_WORKERS)
    print(f"sampled {len(jobs)} designs twice (second run = noise check) in {time.time() - _t0:.0f}s")
    _results_path = _report_root / f"eu_and_v3_results_{datetime.datetime.now():%Y%m%dT%H%M%S}.json"
    _results_path.write_text(json.dumps({"results": results, "check": check_results}))
    print(f"wrote {_results_path}")
else:
    _saved = json.loads(Path(RESULTS_TO_LOAD).read_text())
    results, check_results = _saved["results"], _saved["check"]
    if CONTROLS_SAMPLED and any("run_nt_sampled" not in _c for _run in (results, check_results) for _res in _run.values()
           for _c in _res.get("controls", [])):
        raise ValueError(f"{RESULTS_TO_LOAD} has controls without a sampled Open: it was sampled before the controls "
                         "were sampled; run the sampling again")
    print(f"loaded results for {len(results)} designs from {RESULTS_TO_LOAD}")

## Open, checks and ranking

Per state: `run_nt_sampled` is the mean over the draws of the longest open run r (this is Open),
`open_nt_sampled` the mean number of open nt k, `open_nt_matrix` the exact mean number of open nt, `open_nt_sd` the
standard deviation of k over the draws. The matrix cannot give Open, so a state is *verified* when (1) the sampled mean k
matches the matrix within `VERIFY_SIGMA` standard errors (+ `VERIFY_FLOOR_NT` nt), which shows the sampler is right for that
state, and (2) Open does not exceed the matrix mean k (r <= k always). The check changes no value; its outcome is listed in Appendix A of the report.

Controls: for each design `N_CONTROLS` mismatched triggers. By default (`CONTROLS_SAMPLED = True`) they are sampled like every other state
and checked against the matrix (Appendix A). With `CONTROLS_SAMPLED = False` the exact matrix E[k] of the switch with each is an upper bound of its Open, and the largest is
the Control.

ON = 100 x Open(switch + trigger) / L and OFF = 100 x max(Open(switch alone), Control) / L, both in % of the stem.
Designs are ranked by ON - OFF (percentage points), largest first. Designs within the 95th percentile of the seed-to-seed change
of ON - OFF (second run, seed + `CHECK_SEED_OFFSET`) of the best form the tied group, which is listed first and ordered by dG
= G(switch + trigger) - G(switch) - G(trigger alone). The ratio ON / max(OFF, `OFF_FLOOR_PCT`) is listed for information and orders nothing.


In [ ]:
def _state_ok(state, stem_len):
    # tolerance tau = VERIFY_SIGMA * sd(k) / sqrt(N) + VERIFY_FLOOR_NT; a measured sd of 0 is kept
    # (only a missing sd falls back to L / 2), see _report_stats.verification_tolerance
    return state_ok(state, stem_len, sigma=VERIFY_SIGMA, floor_nt=VERIFY_FLOOR_NT)


scored = []
for _d in designs:
    _r = results[_d["rank"]]
    _L = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
    scored.append({
        "design": _d, "csv_rank": int(_d["rank"]), "stem_len": _L,
        "open_t": _r["on"]["run_nt_sampled"], "k_t": _r["on"]["open_nt_sampled"],
        "k_t_matrix": _r["on"]["open_nt_matrix"], "ok_t": _state_ok(_r["on"], _L),
        # the Control in nt: the worst sampled Open of the controls, or (default) the worst exact matrix E[k], an
        # upper bound of the Open of a mismatched-trigger state
        "ctrl_nt": (max((c["run_nt_sampled"] for c in _r.get("controls", []) if c["run_nt_sampled"] is not None),
                        default=None) if CONTROLS_SAMPLED
                    else max((c["open_nt_matrix"] for c in _r.get("controls", [])), default=None)),
        "n_controls": len(_r.get("controls", [])),
        "n_controls_ok": (sum(_state_ok(c, _L) for c in _r.get("controls", [])) if CONTROLS_SAMPLED else None),
        "open_a": _r["alone"]["run_nt_sampled"], "k_a": _r["alone"]["open_nt_sampled"],
        "k_a_matrix": _r["alone"]["open_nt_matrix"], "ok_a": _state_ok(_r["alone"], _L),
    })

# ON and OFF as % of the stem; a missing Open makes them (and the difference) None, never 0
for _r in scored:
    _ok = _r["open_t"] is not None and _r["open_a"] is not None
    _r["on_pct"] = 100.0 * _r["open_t"] / _r["stem_len"] if _r["open_t"] is not None else None
    _r["off_alone_pct"] = 100.0 * _r["open_a"] / _r["stem_len"] if _r["open_a"] is not None else None
    # control: the worst (largest) of the controls, see ctrl_nt
    _r["ctrl_pct"] = 100.0 * _r["ctrl_nt"] / _r["stem_len"] if _r["ctrl_nt"] is not None else None
    # OFF = the worst (largest) of the alone state and the controls
    _r["off_pct"] = (max(_r["off_alone_pct"], _r["ctrl_pct"] or 0.0)
                     if _r["off_alone_pct"] is not None else None)
    _r["off_source"] = "control" if (_r["ctrl_pct"] or 0.0) > (_r["off_alone_pct"] or 0.0) else "alone"
    _r["diff_pp"] = _r["on_pct"] - _r["off_pct"] if _ok else None
    _r["ratio"] = _r["on_pct"] / max(_r["off_pct"], OFF_FLOOR_PCT) if _ok else None
    _r["ratio_floored"] = _ok and _r["off_pct"] < OFF_FLOOR_PCT
    # dG: G(switch + trigger) - G(switch alone) - G(trigger alone), ensemble free energies (kcal/mol), exact
    _d = _r["design"]
    _r["dg"] = (folder.partition(f"{_d['switch']}&{_d['trigger']}") - folder.partition(_d["switch"])
                - folder.partition(_d["trigger"]))


# rank by ON - OFF, largest first; a design whose difference is None sorts last
scored.sort(key=lambda r: (r["diff_pp"] is None, -(r["diff_pp"] or 0.0), r["csv_rank"]))
for _pos, _r in enumerate(scored, start=1):
    _r["our_rank"] = _pos

# seed-to-seed noise of ON - OFF: the same designs sampled again with another seed
for _r in scored:
    _c = check_results[_r["design"]["rank"]]
    _on2, _off2 = _c["on"]["run_nt_sampled"], _c["alone"]["run_nt_sampled"]
    # sampled controls differ between the two seeds; matrix bounds do not
    _ctrl2_pct = (100.0 * max((x["run_nt_sampled"] for x in _c.get("controls", []) if x["run_nt_sampled"] is not None),
                              default=0.0) / _r["stem_len"] if CONTROLS_SAMPLED else (_r["ctrl_pct"] or 0.0))
    _r["diff_pp_check"] = (100.0 * _on2 / _r["stem_len"]
                           - max(100.0 * _off2 / _r["stem_len"], _ctrl2_pct)
                          if _on2 is not None and _off2 is not None else None)
_noise = sorted(abs(r["diff_pp"] - r["diff_pp_check"]) for r in scored
                if r["diff_pp"] is not None and r["diff_pp_check"] is not None)
noise_median = statistics.median(_noise)
noise_p95 = _noise[int(0.95 * (len(_noise) - 1))]
n_tied_with_best = sum(1 for r in scored
                       if r["diff_pp"] is not None and scored[0]["diff_pp"] - r["diff_pp"] <= noise_p95)
# near-ties: designs within the noise margin of the best are one tied group, ordered by dG
# (more negative first); the rest follow by ON - OFF
_tied = [r for r in scored if r["diff_pp"] is not None and scored[0]["diff_pp"] - r["diff_pp"] <= noise_p95]
_rest = [r for r in scored if r not in _tied]
_tied.sort(key=lambda r: (r["dg"], r["csv_rank"]))
scored = _tied + _rest
for _pos, _r in enumerate(scored, start=1):
    _r["our_rank"] = _pos
    _r["tied"] = _r in _tied
print(f"seed-to-seed |change of ON - OFF|: median {noise_median:.2f}, 95th percentile {noise_p95:.2f} pp; "
      f"{n_tied_with_best} designs within that of rank 1")

n_ok_t = sum(r["ok_t"] for r in scored)
n_ok_a = sum(r["ok_a"] for r in scored)
print(f"{len(scored)} designs; verified: switch + trigger {n_ok_t}, switch alone {n_ok_a}")
print("top 5 (our rank, CSV rank, ON - OFF):",
      [(r["our_rank"], r["csv_rank"], round(r["diff_pp"], 1)) for r in scored[:5]])

## Correlation with the CSV rank (saved as a separate PNG, not part of the report)

Spearman rho is the Pearson correlation of the two rank vectors; Kendall tau-b is
(C - D) / sqrt((n0 - n1)(n0 - n2)) over all pairs. Both are +1 when the orders agree.


In [ ]:
def _average_ranks(values):
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2 + 1
        i = j + 1
    return ranks


def spearman_rho(x, y):
    rx, ry = _average_ranks(x), _average_ranks(y)
    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)
    cov = sum((a - mx) * (b - my) for a, b in zip(rx, ry, strict=True))
    vx, vy = sum((a - mx) ** 2 for a in rx), sum((b - my) ** 2 for b in ry)
    return cov / math.sqrt(vx * vy) if vx and vy else None


def kendall_tau_b(x, y):
    n = len(x)
    concordant = discordant = ties_x = ties_y = 0
    for i in range(n):
        for j in range(i + 1, n):
            dx, dy = x[i] - x[j], y[i] - y[j]
            if dx == 0 and dy == 0:
                continue
            if dx == 0:
                ties_x += 1
            elif dy == 0:
                ties_y += 1
            elif (dx > 0) == (dy > 0):
                concordant += 1
            else:
                discordant += 1
    denom = math.sqrt((concordant + discordant + ties_x) * (concordant + discordant + ties_y))
    return (concordant - discordant) / denom if denom else None


_ranked = [r for r in scored if r["diff_pp"] is not None]
csv_ranks = [r["csv_rank"] for r in _ranked]
our_ranks = [r["our_rank"] for r in _ranked]
rho = spearman_rho(csv_ranks, our_ranks)
tau = kendall_tau_b(csv_ranks, our_ranks)
print(f"Spearman rho = {rho:.3f}   Kendall tau-b = {tau:.3f}   over {len(_ranked)} designs")

In [ ]:
import io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm"})
fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.scatter(csv_ranks, our_ranks, s=14, c="#7f8c8d", alpha=0.7, label="all designs")
_top = [r for r in _ranked if r["our_rank"] <= REPORT_TOP_N]
ax.scatter([r["csv_rank"] for r in _top], [r["our_rank"] for r in _top], s=22, c="#c0392b",
           label=f"top {REPORT_TOP_N} (listed in the report)")
ax.plot([1, len(_ranked)], [1, len(_ranked)], "k--", lw=0.8, label="same rank in both")
ax.set_xlabel("rank in the CSV (1 = best)")
ax.set_ylabel("rank by ON - OFF, this report (1 = best)")
ax.invert_yaxis()
ax.set_title(f"Spearman rho = {rho:.2f},  Kendall tau-b = {tau:.2f}")
ax.legend(fontsize=8, loc="lower left")
ax.grid(alpha=0.3)
fig.tight_layout()
_buf = io.BytesIO()
fig.savefig(_buf, format="png", dpi=130)
plt.close(fig)
_corr_png_path = _report_root / f"eu_and_v3_correlation_{datetime.datetime.now():%Y%m%dT%H%M%S}.png"
_corr_png_path.write_bytes(_buf.getvalue())
print(f"wrote correlation plot (outside the report) -> {_corr_png_path}")

## Details for each listed design

For each design in the report table: its parts (domains of the switch and the trigger), for the switch
alone and for the switch with its trigger a 2D picture of the minimum-free-energy structure and the
base-pair-probability matrix, and the controls' E[k]. The structure is the single most stable one, drawn from `FoldEngine.mfe`;
the matrix is the exact `FoldEngine.base_pair_probabilities`. The report (a PDF) has them as an appendix.


In [ ]:
from _report_figures import DOMAIN_COLORS, bpp_heatmap_png, domain_fill, structure_svg


def _fmt_nt(x):
    return "n/a" if x is None else f"{x:.2f}"


_DOMAIN_ORDER = ("prefix", "toehold", "stem_up", "loop", "stem_down", "pre_kozak", "kozak")


def _switch_line_html(design):
    # the whole switch sequence 5' to 3' through the Kozak, each part shaded in its own colour
    kozak_end = design["domains"]["kozak"][1]
    pieces, legend = [], []
    for name in _DOMAIN_ORDER:
        start, end = design["domains"].get(name, (0, 0))
        if end > start:
            colour = DOMAIN_COLORS.get(name, "#C7D0CA")
            pieces.append(f"<span class='dom' style='background:{colour}55' title='{name}'>"
                          f"{design['switch'][start:end]}</span>")
            legend.append(f"<span class='sw' style='background:{colour}'></span>{name}")
    return (f"<p class='note'>Switch sequence, 5&prime; to 3&prime; through the Kozak ({kozak_end} nt):</p>"
            f"<p class='seqline'>{''.join(pieces)}</p><p class='note'>{' &ensp; '.join(legend)}</p>"
            f"<p class='note'>Trigger ({len(design['trigger'])} nt): <span class='seqline'>{design['trigger']}</span></p>")


def _domain_table_html(design):
    switch, trigger = design["switch"], design["trigger"]
    rows = []
    for name in _DOMAIN_ORDER:
        start, end = design["domains"].get(name, (0, 0))
        if end > start:
            rows.append(
                f"<tr><td>{name}</td><td class='n'>{start + 1}&ndash;{end}</td>"
                f"<td class='n'>{end - start}</td><td class='seq'>{switch[start:end]}</td></tr>"
            )
    rows.append(f"<tr><td>trigger</td><td class='n'>&ndash;</td><td class='n'>{len(trigger)}</td>"
                f"<td class='seq'>{trigger}</td></tr>")
    return ("<table class='booktabs small'><tr><th>part</th><th class='n'>positions (1-based)</th>"
            "<th class='n'>length (nt)</th><th>sequence (5&prime; to 3&prime;)</th></tr>"
            + "".join(rows) + "</table>")


def _state_figures(design, strands, heading):
    # (2D structure SVG, base-pair-probability PNG as base64) of one state
    switch = design["switch"]
    sequence = strands.replace("&", "")
    structure = folder.mfe(strands).structure.replace("&", "")
    kozak_end = design["domains"]["kozak"][1] if design["domains"].get("kozak") else len(switch)
    fills = [f for f in domain_fill(design["domains"]) if f[0] < kozak_end]
    svg = structure_svg(
        folder.layout_coordinates(structure), sequence, structure,
        f"{heading} (minimum-free-energy structure)", domain_fill=fills,
        visible_len=kozak_end, construct_len=len(switch),
    )
    boundary = len(switch) if "&" in strands else None
    png = bpp_heatmap_png(sequence, folder.base_pair_probabilities(strands),
                          f"{heading}: base-pair probabilities", boundary)
    return svg, png


for _r in scored[:REPORT_TOP_N]:
    _d = _r["design"]
    _L = _r["stem_len"]
    _on_text = (f"Open = {_fmt_nt(_r['open_t'])} nt; E[k] = {_fmt_nt(_r['k_t'])} nt "
                f"(matrix {_fmt_nt(_r['k_t_matrix'])}); L = {_L}")
    if _r["n_controls"]:
        _ctrl_what = (f"Control = {_fmt_nt(_r['ctrl_nt'])} nt "
                      f"({pct_text(_r['ctrl_pct'])} of L; {_r['n_controls_ok']} of {_r['n_controls']} control states pass the sampler check, Appendix A)"
                      if CONTROLS_SAMPLED else
                      f"worst E[k] from the matrix over {_r['n_controls']} mismatched triggers = {_fmt_nt(_r['ctrl_nt'])} nt "
                      f"({pct_text(_r['ctrl_pct'])} of L; an upper bound of Open)")
        _ctrl_text = (f"{_ctrl_what}; OFF = larger of this and Open(alone) = {pct_text(_r['off_pct'])} ({_r['off_source']}); "
                      f"mismatched triggers = triggers of CSV rows {', '.join(str(x) for x in _d.get('control_ranks', []))}; L = {_L}")
    else:
        _ctrl_text = (f"no mismatched trigger was available for this design (none shares at most {CONTROL_MAX_OVERLAP_NT} nt of identity "
                      f"with its trigger), so there is no control; OFF = Open(alone) = {pct_text(_r['off_pct'])}; L = {_L}")
    _off_text = (f"Open = {_fmt_nt(_r['open_a'])} nt; E[k] = {_fmt_nt(_r['k_a'])} nt "
                 f"(matrix {_fmt_nt(_r['k_a_matrix'])}); L = {_L}")
    _svg_off, _png_off = _state_figures(_d, _d["switch"], "OFF: switch alone")
    _svg_on, _png_on = _state_figures(_d, _d["switch"] + "&" + _d["trigger"], "ON: switch + trigger")
    _r["detail_html"] = (
        f"<div class='panel' id='p{_r['our_rank']}'><h3>Design rank {_r['our_rank']} "
        f"&middot; stem L = {_L} nt "
        f"<a class='back' href='#results'>&uarr; back to the table</a></h3>"
        f"{_switch_line_html(_d)}{_domain_table_html(_d)}"
        f"<div class='state'><h4>OFF: switch alone</h4><p class='note'>{_off_text}</p>{_svg_off}</div>"
        f"<div class='state'><h4>ON: switch + trigger</h4><p class='note'>{_on_text}</p>{_svg_on}</div>"
        f"<p class='note'><b>Controls: switch + mismatched triggers.</b> {_ctrl_text}</p>"
        f"<div class='bppRow'>"
        f"<figure><img src='data:image/png;base64,{_png_off}'>"
        f"<figcaption>OFF: switch alone</figcaption></figure>"
        f"<figure><img src='data:image/png;base64,{_png_on}'>"
        f"<figcaption>ON: switch + trigger</figcaption></figure>"
        f"</div></div>"
    )
print(f"details built for {REPORT_TOP_N} designs")

## Report (PDF and CSV of part 1)

The report is a PDF: it is built from one HTML template, rendered with `render_pdf.py`, and the HTML is deleted afterwards (with `MAKE_PDF = False` the HTML is kept instead).


In [ ]:
def _fmt_nt(x):
    return "n/a" if x is None else f"{x:.2f}"


def _fmt_nt_pct(x, length):
    return "n/a" if x is None else f"{x:.2f}&nbsp;({100.0 * x / length:.0f}%)"


_CSS = r'''
@import url("https://cdn.jsdelivr.net/gh/aaaakshat/cm-web-fonts@latest/fonts.css");
@page { size: A4; margin: 20mm 18mm; }
body { font-family: "Computer Modern Serif", "CMU Serif", "Latin Modern Roman", "Times New Roman", serif;
       font-size: 11.5pt; line-height: 1.5; color: #111; max-width: 174mm; margin: 0 auto; }
h1 { font-size: 20pt; font-weight: normal; text-align: center; margin: 0 0 4pt 0; line-height: 1.25; }
.byline { text-align: center; font-size: 10pt; color: #444; margin-bottom: 18pt; }
h2 { font-size: 13.5pt; margin: 22pt 0 6pt 0; font-weight: bold; break-after: avoid; }
p { margin: 6pt 0; text-align: justify; }
dl { margin: 6pt 0; } dt { break-after: avoid; font-weight: bold; float: left; clear: left; width: 34mm; }
dd { break-inside: avoid; margin: 0 0 4pt 36mm; }
code { font-family: "Courier New", monospace; font-size: 9.5pt; }
table { border-collapse: collapse; margin: 8pt auto; font-size: 6.8pt; }
table.booktabs { border-top: 1.5pt solid #111; border-bottom: 1.5pt solid #111; }
table.booktabs th { border-bottom: 0.75pt solid #111; font-weight: bold; padding: 3pt 2pt; text-align: left; }
table.booktabs th.n { white-space: nowrap; }
table.booktabs td { padding: 2pt 2pt; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; }
.ok { color: #1e7e34; } .no { color: #b02a37; font-weight: bold; }
figure { margin: 10pt 0; text-align: center; break-inside: avoid; }
figcaption { font-size: 10pt; color: #333; margin-top: 4pt; text-align: left; }
.note { font-size: 10pt; color: #333; }
.katex { font-size: 1.0em !important; } .katex-display { margin: 0.55em 0 !important; }
th.grp { text-align: center; border-bottom: 0.75pt solid #111; }
th.on { background: #fbe9e6; } th.off { background: #e6eef7; } th.ctl { background: #eef3e6; }
tr.detail > td { padding: 8pt 4pt 14pt 4pt; border-bottom: 0.75pt solid #999; }
.panel h3 { font-size: 12pt; margin: 0 0 6pt 0; } .state h4 { font-size: 11pt; margin: 6pt 0 2pt 0; }
.state { margin-top: 10pt; }
.bppRow { display: flex; gap: 12pt; justify-content: center; margin-top: 10pt; break-inside: avoid; }
.bppRow figure { flex: 1; margin: 2pt; text-align: center; } .bppRow img { max-width: 100%; }
td.seq { font-family: "Courier New", monospace; font-size: 8.5pt; word-break: break-all; }
table.small { font-size: 8.5pt; margin: 4pt 0; }
@media print { .detail, .detail-btn, .noprint { display: none !important; } }
.detail-link, .back, .detail-btn { white-space: nowrap; font-family: inherit; font-size: 9.5pt; color: #1a5fb4;
  text-decoration: underline; font-weight: normal; border: none; background: none; padding: 0; cursor: pointer; }
.back { margin-left: 14pt; }
.seqline { font-family: "Courier New", monospace; font-size: 9pt; word-break: break-all; line-height: 1.7; }
.dom { padding: 1pt 0; } .sw { display: inline-block; width: 8pt; height: 8pt; margin-right: 3pt; }
.abstract { border-left: 3pt solid #1F6FEB; padding: 2pt 8pt; margin: 8pt 0 12pt 0; font-size: 11pt; background: #f4f7fb; }
.appendix2 { break-before: page; } .appendix .panel { break-before: page; } .appendix .panel:first-of-type { break-before: auto; } h2 + table { break-inside: avoid; } table.small, table.small tr { break-inside: avoid; } #results + p { break-after: avoid; break-inside: avoid; } .appendix .state { break-inside: avoid; }
.appendix svg { max-height: 112mm !important; }
figure.construct svg { width: 100%; max-width: 165mm; height: auto; }
figure.construct figcaption { text-align: left; }
'''

_TEMPLATE = r'''<!doctype html><html><head><meta charset="utf-8"><title>eu_and_v3 report</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/contrib/auto-render.min.js"></script>
<script>document.addEventListener("DOMContentLoaded", function () {
  renderMathInElement(document.body, {delimiters: [
    {left: "\\[", right: "\\]", display: true}, {left: "\\(", right: "\\)", display: false}]});
});</script>
<style>@@CSS@@</style></head><body>

<h1>Single-switch screen</h1>
<div class="byline">@@SOURCE@@<br>First @@N_DESIGNS@@ designs (the table lists the best @@REPORT_TOP_N@@), folded at
@@TEMPERATURE@@&nbsp;&deg;C, @@SALT@@&nbsp;M salt, with @@ENGINE@@ &middot; @@DATE@@</div>

<div class="abstract"><b>Summary.</b> @@ABSTRACT@@</div>

<h2>1&ensp;Definitions</h2>
<figure class="construct">@@FIG_CONSTRUCT@@<figcaption><b>Figure 1.</b> The parts of a switch, drawn on the fused construct of the AND screen: a single switch is one of these hairpins (the left one, blue) with its own prefix, toehold, loop, Kozak and payload, and one trigger. <i>Top:</i> the switch alone, hairpin closed. <i>Bottom:</i> the trigger bound to the toehold and \(\mathrm{stem}_{\uparrow}\), the stem open. (The second hairpin and the spacer belong to the AND screen only.)</figcaption></figure>
<dl>
<dt>switch</dt><dd>the RNA switch strand of a design. It contains one hairpin (Figure 1).</dd>
<dt>trigger</dt><dd>the design's real trigger RNA.</dd>
<dt>stem</dt><dd>(Figure 1) the hairpin stem has two sides of \(L\) nucleotides each, \(\mathrm{stem}_{\uparrow}\) (5&prime; side)
and \(\mathrm{stem}_{\downarrow}\) (3&prime; side). In the closed hairpin they pair with each other.
Here \(L\) = @@STEM_LENS@@.</dd>
<dt>state</dt><dd>a set of strands folded together: <i>switch + trigger</i> (the switch with its real trigger),
<i>switch alone</i> (no trigger) or a <i>control</i> (the switch with a mismatched trigger).</dd>
<dt>mismatched trigger</dt><dd>the real trigger of another design of the same table (or, if none qualifies, another sequence offered for this purpose) that shares at most @@CONTROL_OVERLAP@@ nt of identical sequence with the true trigger; @@N_CONTROLS@@ of them per design are used as wrong inputs.</dd>
<dt>prefix, Kozak, payload</dt><dd>(Figure 1) the switch RNA starts with a short <i>prefix</i>, then the toehold and the hairpin; the <i>Kozak</i> sequence follows the
hairpin and the <i>payload</i> (the coding sequence, 67 nt here) follows the Kozak. The payload is folded together with the rest but is not drawn.</dd>
<dt>how to read the figures</dt><dd>Figure 1 is a schematic; the other figures are computed. 2D structures: every circle is a nucleotide coloured by part; a <b>gold ring</b> marks a nucleotide paired into the payload
(the undrawn &ldquo;hidden tail&rdquo; after the Kozak) and a <b>pink ring</b> one paired to the (undrawn) trigger strand. Heat maps: the axes run over the
whole folded RNA, including the payload and, for the switch + trigger state, the trigger; the dotted red lines mark where one strand ends and the next begins.</dd>
<dt>toehold, hairpin, loop</dt><dd>(Figure 1) the <i>toehold</i> is the single-stranded stretch of the switch where the trigger first binds; the <i>hairpin</i> is the stem (\(\mathrm{stem}_{\uparrow}\) paired with \(\mathrm{stem}_{\downarrow}\)) closed by a <i>loop</i> of unpaired nucleotides. Binding the trigger is meant to open the hairpin.</dd>
<dt>pp</dt><dd>percentage points, the unit of a difference of two percentages.</dd>
<dt>structure</dt><dd>one folded arrangement \(s\) of the strands, i.e. a set of base pairs.</dd>
<dt>\(E(s)\), \(G\)</dt><dd>the energy of structure \(s\), and the free energy of the whole ensemble of
structures of the state (both in kcal/mol).</dd>
<dt>\(\Delta G\)</dt><dd>\(G_{\text{switch + trigger}}-G_{\text{switch alone}}-G_{\text{trigger alone}}\), in kcal/mol, from the exact partition
function: the free energy gained by forming the complex from the two separate molecules. More negative means the trigger binds the switch more strongly (no concentration term). It is listed for information and orders only the tied group; a more stable hairpin makes it less negative, so the ordering slightly favours less stable switches. Strand concentration is not included (the duplex-initiation term of @@ENGINE_NAME@@'s dimer energy is).</dd>
<dt>\(P(s)\)</dt><dd>the probability of \(s\) in the ensemble, \(P(s)=\exp\!\big(-(E(s)-G)/RT\big)\), with \(R\) the gas constant and \(T\) the absolute temperature.</dd>
<dt>open nucleotide</dt><dd>a \(\mathrm{stem}_{\uparrow}\) nucleotide that is <i>not</i> paired to a
\(\mathrm{stem}_{\downarrow}\) nucleotide in \(s\) (any one, including a shifted partner). It may be unpaired, or paired to the trigger or to anything else.</dd>
<dt>\(k(s)\)</dt><dd>the number of open nucleotides in \(s\), from \(0\) to \(L\), anywhere on the stem.</dd>
<dt>\(r(s)\)</dt><dd>the length of the longest stretch of <i>consecutive</i> open nucleotides in \(s\),
from \(0\) to \(L\). Always \(r(s)\le k(s)\). Nucleotides in one stretch are open at the same time.</dd>
</dl>

<h2>2&ensp;Open</h2>
<p>For a state, <b>Open</b> is the expected length of the <b>longest run of consecutive open nucleotides</b>
(not the total number of open nucleotides),
\[ \mathrm{Open}\;=\;\mathbb{E}[r]\;=\;\sum_{s}P(s)\,r(s), \]
reported in nucleotides and as a percentage \(100\cdot\mathrm{Open}/L\) of the stem. It is computed for two states:
\[ \begin{aligned}
\mathrm{Open}_{\text{trigger}}&=\mathbb{E}[r]\ \text{of the switch with its real trigger},\\
\mathrm{Open}_{\text{alone}}&=\mathbb{E}[r]\ \text{of the switch alone}.
\end{aligned} \]
A good switch has \(\mathrm{Open}_{\text{trigger}}\) large (the hairpin opens when its trigger is present) and
\(\mathrm{Open}_{\text{alone}}\) small (it stays closed without it).
</p>
<p>Both are expressed as a percentage of the stem and compared,
\[ \begin{aligned}
&\mathrm{ON}=100\,\mathrm{Open}_{\text{trigger}}/L,\qquad \mathrm{OFF}=100\,\max\big(\mathrm{Open}_{\text{alone}},\,\mathrm{Control}\big)/L,\\[1pt]
&\mathrm{ON}-\mathrm{OFF}\ \text{(percentage points)},\qquad \mathrm{ratio}=\mathrm{ON}\,/\max(\mathrm{OFF},\,@@FLOOR@@).
\end{aligned} \]
OFF is the <b>worst</b> (largest) of the switch alone and the controls (below). <b>The designs are ranked by \(\mathrm{ON}-\mathrm{OFF}\)</b> in percentage points, largest first, with
near-ties handled as described below. A percentage of the stem makes designs with different stem lengths comparable (a count of
nucleotides would favour long stems for ON. For OFF the effect may reverse (fraying at the stem ends is roughly constant in nucleotides), so stem lengths are not fully neutral. Counts would favour long stems, which have more nucleotides that can open). The ratio is listed for
information and orders nothing; the floor of @@FLOOR@@% on OFF keeps it finite when OFF is close to zero, and a
ratio computed with the floor is marked with an asterisk.</p>
@@CTRL_PARA@@
<p><b>Ties.</b> Repeating the sampling with another seed changes ON &minus; OFF by a small amount (section 5). Designs whose ON &minus; OFF lies within the
95th-percentile of that change from the best one cannot be told apart from it; they form one <i>tied group</i> (marked &dagger; in the table), ranked inside the group by \(\Delta G\) (more negative first, i.e. stronger binding of the trigger) and
listed before the other designs. Only Monte-Carlo noise is counted here; the error of the energy model is larger, so designs just outside the group are not clearly worse either, and tied designs may share a trigger or toehold, so they are not independent candidates. The tied group is the report's shortlist; &dagger; marks it.</p>
<p><b>What Open does and does not measure.</b> The ON state is the ensemble of the switch <i>with its trigger already bound as a complex</i>; the trigger concentration, the binding kinetics and the fraction of switches that actually form the complex are not modelled. Open measures how far the hairpin stem is opened at equilibrium (the exact ensemble of @@ENGINE_NAME@@ at the stated
temperature). It is a structural proxy for the hairpin releasing the region behind it; it is not a measurement of translation, of ribosome binding or
start-codon accessibility, of kinetics, or of behaviour in a cell, and trigger accessibility and self-structure, cross-talk between triggers, the openness of \(\mathrm{stem}_{\downarrow}\) and of the start codon, temperature or parameter-set sensitivity, and the origin of the CSV are not assessed; the controls are at most @@N_CONTROLS@@ mismatched triggers per design (the number actually found for each design is counted in the summary table; with fewer, or none, OFF rests on the alone state and the controls that exist)@@CTRL_LIMIT@@, and there are no scrambled triggers or known reference switch, so the scale of the numbers is not calibrated against experiment.</p>
<p>Next to each Open the report lists \(\mathbb{E}[k]\), the <b>expected number of open nucleotides anywhere on
the stem</b>, whether or not they are consecutive,
\[ \mathbb{E}[k]\;=\;\sum_{s}P(s)\,k(s)\;\ge\;\mathrm{Open}. \]
For the listed designs the two nearly coincide (the open nucleotides are mostly contiguous). It is the more relevant number for a state that should stay closed: a stem that is open in scattered places has a small
Open but may have a larger \(\mathbb{E}[k]\). It is given sampled and from the exact matrix. The stem length
\(L\) of each design is listed too, because Open and \(\mathbb{E}[k]\) are counts out of \(L\).</p>

<h2>3&ensp;Estimation and verification</h2>
<p><i>In plain words:</i> Open is estimated by drawing random structures with the probability they have at equilibrium. The exact base-pair probability matrix cannot give Open but can give the average number of open nucleotides, so it is used to check that the drawn structures behave correctly.</p>
<p><b>Sampled estimate.</b> The sum over all structures cannot be written out, so \(N\) structures
\(S_1,\dots,S_N\) are drawn from the ensemble (@@SAMPLER_METHOD@@) and averaged,
\[ \mathbb{E}[r]\;\approx\;\frac1N\sum_{i=1}^{N}r(S_i),\qquad N=@@N_SAMPLES@@ . \]
A structure is drawn with probability \(P(s)\), so this estimates the full sum, including structures that are
never drawn. <b>No normalisation is needed:</b> the draws are already distributed according to \(P(s)\), and the average
needs no division by the share of the ensemble that the distinct sampled structures cover (that share is small, since
almost every draw is a different structure; dividing by it would only be needed if the exact \(P(s)\) were summed over
the distinct sampled structures instead). Its statistical error is \(\mathrm{sd}(r)/\sqrt N\).</p>
@@SAMPLER_HOW@@
<p><b>Matrix check.</b> The base-pair probability matrix \(p_{ij}\) (the probability that nucleotides \(i\) and \(j\)
are paired, summed over all structures) gives the expected <i>number</i> of open nucleotides exactly,
\[ \mathbb{E}[k]\;=\;\sum_{i\in \mathrm{stem}_{\uparrow}}\Big(1-\sum_{j\in \mathrm{stem}_{\downarrow}}p_{ij}\Big). \]
It cannot give \(\mathbb{E}[r]\), because it does not know which positions are open together. It does give
the bound \(\mathbb{E}[r]\le\mathbb{E}[k]\). A state is therefore <i>verified</i> (it is then listed as verified in Appendix A) when
(1) the sampled \(\mathbb{E}[k]\) agrees with the matrix value to within \(\tau\), which shows the sampler is
right for that state, and (2) \(\mathrm{Open}\le\mathbb{E}[k]_{\text{matrix}}+\tau\) (a sanity bound: \(r\le k\) holds for every draw, so this only catches a broken estimate), with \(\tau\) given below. Passing the check therefore shows that the sampler
reproduces the per-nucleotide openness of that state to within \(\tau\); it does not prove the longest runs themselves, which the matrix cannot see.</p>
<p><b>The tolerance.</b> The sampled mean of \(k\) has a standard error \(\mathrm{sd}(k)/\sqrt N\), where \(\mathrm{sd}(k)\) is the standard deviation of \(k\) over the \(N\) draws of that very state. The tolerance is therefore set per state,
\(\tau=@@VERIFY_SIGMA@@\,\mathrm{sd}(k)/\sqrt N+@@VERIFY_FLOOR@@\) nucleotides, so a state that should stay closed (small \(\mathbb{E}[k]\), small \(\mathrm{sd}\)) is checked tightly and a variable one is checked loosely. The constant @@VERIFY_FLOOR@@ nt only
allows for the rounding of the matrix. Verification catches gross sampler faults; \(\tau\) can be as large as the seed-to-seed noise of the ranking, so passing the check does not guarantee ranking-level accuracy. The tolerance only decides whether a state counts as verified; it changes no Open or \(\mathbb{E}[k]\) value.</p>
@@STRAND_ORDER@@
<p><b>Unverified states.</b> States that fail the check stay in all medians, noise figures and rank correlations; they are listed in Appendix A. The sampler's dependence on strand order is empirical, not explained.</p>
@@SAMPLER_LIMIT@@

<h2>4&ensp;Parameters</h2>
<table class="booktabs"><tr><th>name</th><th>value</th><th>meaning</th></tr>
@@PARAMS@@
</table>

<h2>5&ensp;Summary</h2>
<table class="booktabs">
<tr><td>designs scored</td><td class="n">@@N_SCORED@@</td></tr>
<tr><td>median ON / OFF (% of the stem)</td><td class="n">@@MED_T@@ / @@MED_A@@</td></tr>
<tr><td>median ON &minus; OFF (percentage points)</td><td class="n">@@MED_D@@</td></tr>
<tr><td>@@CTRL_MED@@, % of the stem (n/a when no design has a control)</td><td class="n">@@MED_C@@</td></tr>
<tr><td>designs with the full @@N_CONTROLS@@ controls / fewer / none (OFF is then the alone state, or the fewer controls found)</td><td class="n">@@CTRL_COUNTS@@</td></tr>
<tr><td>seed-to-seed change of ON &minus; OFF, second independent run: median / 95th percentile (pp)</td><td class="n">@@NOISE_MED@@ / @@NOISE_P95@@</td></tr>
</table>
<p>@@BEST_TEXT@@</p>

<h2 id="results">6&ensp;First @@REPORT_TOP_N@@ designs (tied group first, then by \(\mathrm{ON}-\mathrm{OFF}\))</h2>
<p class="note">&dagger; = member of the tied group (section 2); inside it the order is by \(\Delta G\), so rank 1 need not have the highest ON &minus; OFF. * after a ratio = OFF was below the floor, so the floor was used. \(L\) is the stem length of the design. Open is the expected <b>longest run</b> of consecutive open
nucleotides, in nucleotides with the percentage of \(L\) in brackets. The expected number of open nucleotides anywhere on the stem, \(\mathbb{E}[k]\), sampled and exact (matrix), with the sampler check, is in Appendix A at the end. @@CTRL_DEF@@ OFF is the larger of the alone Open and the Control (in %). The last columns are ON &minus; OFF (percentage points of the stem), the ratio ON / OFF and \(\Delta G\) (kcal/mol). The Details link of each row jumps to the design's parts, 2D structures and base-pair-probability matrices in section 7.</p>
<table class="booktabs">
<thead><tr><th rowspan="2" class="n">rank</th><th rowspan="2" class="n">\(L\)</th>
<th class="grp on">ON</th>
<th class="grp off">OFF</th>
<th class="grp ctl">Control</th>
<th rowspan="2" class="n">ON &minus; OFF<br>(pp)</th><th rowspan="2" class="n">ratio<br>ON / OFF</th><th rowspan="2" class="n">\(\Delta G\)<br>(kcal/mol)<br>lower = tighter</th>@@BTNHEAD@@</tr>
<tr><th class="n">\(\mathrm{Open}_{\text{trigger}}\)<br>switch + trigger<br>longest run</th>
<th class="n">\(\mathrm{Open}_{\text{alone}}\)<br>switch alone<br>longest run</th>
<th class="n">@@CTRL_HDR@@</th></tr></thead>
@@ROWS@@
</table>
@@APPENDIX@@
@@CHECK_APPENDIX@@
</body></html>'''

_stem_text = ", ".join(str(n) for n in sorted({r["stem_len"] for r in scored}))
_param_rows = [
    ("N_DESIGNS", N_DESIGNS, "first rows of the CSV that are scored"),
    ("TEMPERATURE_C", TEMPERATURE_C, "folding temperature, &deg;C"),
    ("SALT_MOLAR", SALT_MOLAR, "monovalent salt, mol/L (no magnesium)"),
    ("N_SAMPLES", N_SAMPLES, "structures drawn from the ensemble for each state"),
    ("SEED", SEED, "random seed for the sampler (a rerun repeats)"),
    ("CHECK_SEED_OFFSET", CHECK_SEED_OFFSET, "a second sampling run uses SEED + this; it only measures the noise (section 5)"),
    ("VERIFY_SIGMA", VERIFY_SIGMA, "&tau;: allowed disagreement with the matrix, in standard errors of the sampled mean"),
    ("VERIFY_FLOOR_NT", VERIFY_FLOOR_NT, "added to &tau; (nt)"),
    ("N_CONTROLS", N_CONTROLS, "mismatched triggers per design; OFF is the worst of the alone state and these"),
    ("CONTROL_MAX_OVERLAP_NT", CONTROL_MAX_OVERLAP_NT, "the mismatched control trigger shares at most this many nt of identical sequence with the true trigger"),
    ("OFF_FLOOR_PCT", OFF_FLOOR_PCT, "OFF is not allowed below this many % of the stem in the ratio ON / OFF"),
    ("REPORT_TOP_N", REPORT_TOP_N, "designs listed in the table of section 6"),
]
_params_html = "".join(
    f"<tr><td><code>{n}</code></td><td>{v}</td><td>{m}</td></tr>" for n, v, m in _PROVENANCE_ROWS + _param_rows
)
def _fmt_pp(x):
    return "n/a" if x is None else f"{x:.1f}"


def _fmt_ratio(r):
    if r["ratio"] is None:
        return "n/a"
    return f"{r['ratio']:.0f}" + ("*" if r["ratio_floored"] else "")


def _row_html(r):
    L = r["stem_len"]
    button = f"<td><a class='detail-link' href='#p{r['our_rank']}'>Details</a></td>"
    return (
        f"<tr><td class='n'>{r['our_rank']}{'&dagger;' if r['tied'] else ''}</td><td class='n'>{L}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['open_t'], L)}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['open_a'], L)}</td>"
        f"<td class='n'>{_fmt_nt_pct(r['ctrl_nt'], L)}</td>"
        f"<td class='n'><b>{_fmt_pp(r['diff_pp'])}</b></td><td class='n'>{_fmt_ratio(r)}</td>"
        f"<td class='n'>{r['dg']:.1f}</td>"
        f"{button}</tr>"
    )


_rows_html = "".join(_row_html(r) for r in scored[:REPORT_TOP_N])
_n = len(scored)
_fields = {
    **_WORDS, **_WORDS_SINGLE,
    "CSS": _CSS, "SOURCE": html.escape(Path(CSV_PATH).name),
    "ABSTRACT": (f"Each of the first {_n} designs of the CSV (in the order of the input file, whose own ranking is not described here) is folded with and without its trigger. <b>ON</b> is how far the hairpin stem is open with the trigger, <b>OFF</b> how far without it or with a wrong trigger (the worst of the alone state and up to {N_CONTROLS} mismatched triggers), "
        f"both as a percentage of the stem (100% = fully open). A good switch has ON near 100% and OFF near 0%. Here the median ON is {median_text((r['on_pct'] for r in scored), unit='%')} and the median OFF "
        f"{median_text((r['off_pct'] for r in scored), unit='%')}. Designs are ranked by ON &minus; OFF; {n_tied_with_best} of {_n} lie within the run-to-run difference (95th percentile of the change of ON &minus; OFF between two independent samplings: {noise_p95:.2f} points) of the best and cannot be told apart by that score. "
        f"They are marked &dagger; and are the shortlist; among them pick by \\(\\Delta G\\) (the order of the table) or by the details. This is a structure-based prediction only, with no experimental calibration." + _SAMPLER_BANNER), "N_DESIGNS": _n, "TEMPERATURE": f"{TEMPERATURE_C:g}",
    "DATE": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), "STEM_LENS": _stem_text,
    "N_SAMPLES": N_SAMPLES, "VERIFY_SIGMA": f"{VERIFY_SIGMA:g}", "VERIFY_FLOOR": f"{VERIFY_FLOOR_NT:g}",
    "PARAMS": _params_html, "N_SCORED": _n,
    "MED_T": median_text((r['on_pct'] for r in scored), unit='%'),
    "MED_A": median_text((r['off_pct'] for r in scored), unit='%'),
    "MED_D": median_text(r['diff_pp'] for r in scored),
    "FLOOR": f"{OFF_FLOOR_PCT:g}", "CONTROL_OVERLAP": CONTROL_MAX_OVERLAP_NT, "N_CONTROLS": N_CONTROLS,
    "MED_C": median_text((r['ctrl_pct'] for r in scored), unit='%'),
    "CTRL_COUNTS": (f"{sum(r['n_controls'] >= N_CONTROLS for r in scored)} / "
                    f"{sum(0 < r['n_controls'] < N_CONTROLS for r in scored)} / {sum(r['n_controls'] == 0 for r in scored)}"),
    "NOISE_MED": f"{noise_median:.2f}", "NOISE_P95": f"{noise_p95:.2f}",
    "BEST_TEXT": (f"Highest ON &minus; OFF: {max(r['diff_pp'] for r in scored if r['diff_pp'] is not None):.1f} points. "
                  f"{n_tied_with_best} of the {len(scored)} designs lie within the 95th-percentile seed-to-seed change "
                  f"({noise_p95:.2f} points) of it. They form the tied group (&dagger;), which is the shortlist; inside it designs are ordered by \\(\\Delta G\\), not by ON &minus; OFF."),
    "REPORT_TOP_N": REPORT_TOP_N, "ROWS": _rows_html,
}
def _fill(template, fields):
    for _key, _value in fields.items():
        template = template.replace(f"@@{_key}@@", str(_value))
    return template


_appendix = ("<div class='appendix'><h2>7&ensp;Design details</h2><p class='note'>About two pages per design of the "
             "table above (the 2D structures of the switch alone and of the switch with its trigger, then their base-pair probabilities side by side): its parts, and the minimum-free-energy structure and base-pair probabilities of "
             "each state.</p>"
             + "".join(r["detail_html"] for r in scored[:REPORT_TOP_N]) + "</div>")
_unverified_single = "; ".join(
    f"design {r['our_rank']} (CSV rank {r['csv_rank']}): "
    + " and ".join(name for name, ok in (("switch + trigger", r["ok_t"]), ("switch alone", r["ok_a"])) if not ok)
    for r in scored if not (r["ok_t"] and r["ok_a"])) or "none"


def _check_row(r):
    # one row per listed design and state: sampled vs exact E[k] and the verdict, no colour
    cells = []
    for label, k, k_matrix, ok in (("switch + trigger", r["k_t"], r["k_t_matrix"], r["ok_t"]),
                                   ("switch alone", r["k_a"], r["k_a_matrix"], r["ok_a"])):
        cells.append(f"<tr><td class='n'>{r['our_rank']}</td><td class='n'>{r['csv_rank']}</td><td>{label}</td>"
                     f"<td class='n'>{_fmt_nt(k)}</td><td class='n'>{_fmt_nt(k_matrix)}</td>"
                     f"<td>{'&#10003;' if ok else '<b>&times; not verified</b>'}</td></tr>")
    if CONTROLS_SAMPLED and r["n_controls"]:
        failed = r["n_controls"] - r["n_controls_ok"]
        cells.append(f"<tr><td class='n'>{r['our_rank']}</td><td class='n'>{r['csv_rank']}</td>"
                     f"<td>mismatched triggers ({r['n_controls']})</td><td class='n'>&ndash;</td><td class='n'>&ndash;</td>"
                     f"<td>{r['n_controls_ok']} of {r['n_controls']} verified" + (f" (<b>{failed} &times;</b>)" if failed else "") + "</td></tr>")
    return "".join(cells)


_check_appendix = (
    "<div class='appendix2'><h2 id='appendixA'>Appendix A&ensp;Sampler check</h2>"
    "<p class='note'>For each state the sampled \\(\\mathbb{E}[k]\\), the expected number of open nucleotides anywhere on the stem, "
    "is compared with the exact value from the base-pair probability matrix; a state is verified when they agree within the tolerance "
    "of section 3 and Open does not exceed the matrix value by more than that tolerance. It changes no value of the report. "
    f"In this run {n_ok_t} of {_n} switch + trigger states and {n_ok_a} of {_n} switch-alone states passed. "
    f"States not verified (design rank in this report, CSV rank): {_unverified_single}. "
    + (f"The control states (switch with a mismatched trigger) are sampled and checked the same way: {sum(r['n_controls_ok'] for r in scored)} of "
       f"{sum(r['n_controls'] for r in scored)} passed; the table gives the count per listed design. " if CONTROLS_SAMPLED else
       "The control states are not sampled: each is taken from the exact matrix (section 2), so there is nothing to check. ") +

    f"The table lists the first {REPORT_TOP_N} designs; the CSV has the verdicts of all designs.</p>"
    "<table class='booktabs'><thead><tr><th class='n'>rank</th><th class='n'>CSV rank</th><th>state</th>"
    "<th class='n'>\\(\\mathbb{E}[k]\\)<br>sampled</th><th class='n'>\\(\\mathbb{E}[k]\\)<br>matrix</th><th>verified</th></tr></thead>"
    + "".join(_check_row(r) for r in scored[:REPORT_TOP_N]) + "</table></div>")
# the report is a PDF only; this HTML is its source (deleted once the PDF exists)
_print_fields = {**_fields, "BTNHEAD": '<th rowspan="2"></th>', "APPENDIX": _appendix, "CHECK_APPENDIX": _check_appendix}
report_print_html = _fill(_TEMPLATE, _print_fields)

_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
_print_html_path = _report_root / f"eu_and_v3_report_{_ts}.html"
_print_html_path.write_text(report_print_html)

import csv as _csv
all_csv_path = _report_root / f"eu_and_v3_all_designs_{_ts}.csv"
with all_csv_path.open("w", newline="") as _fh:
    _w = _csv.writer(_fh)
    _w.writerow(["our_rank", "csv_rank", "stem_len", "Open_trigger_nt", "Ek_trigger_sampled_nt",
                 "Ek_trigger_matrix_nt", "trigger_verified", "Open_alone_nt", "Ek_alone_sampled_nt",
                 "Ek_alone_matrix_nt", "alone_verified", "ON_pct", "OFF_pct", "ON_minus_OFF_pp",
                 "ratio_ON_over_OFF", "dG_kcal_per_mol", "tied_group", "OFF_alone_pct",
                 _CTRL_CSV, "OFF_source", "n_controls", "n_controls_verified", "control_trigger_csv_ranks"])
    for r in scored:
        _w.writerow([r["our_rank"], r["csv_rank"], r["stem_len"], r["open_t"], r["k_t"],
                     r["k_t_matrix"], r["ok_t"], r["open_a"], r["k_a"], r["k_a_matrix"], r["ok_a"],
                     r["on_pct"], r["off_pct"], r["diff_pp"], r["ratio"], r["dg"], r["tied"], r["off_alone_pct"], r["ctrl_pct"],
                     r["off_source"], r["n_controls"], r["n_controls_ok"], " ".join(str(x) for x in r["design"].get("control_ranks", []))])
print(f"wrote {all_csv_path}")


def _render_pdf(html_path):
    if not MAKE_PDF:
        print("MAKE_PDF is False -- the HTML source of the PDF is kept instead")
        return None
    for base in (Path.cwd(), *Path.cwd().parents):
        script = base / "src/engine/gates/notebooks/toehold/render_pdf.py"
        if script.exists():
            break
    else:
        print("render_pdf.py not found -- HTML written, PDF skipped")
        return None
    pdf_path = html_path.with_suffix(".pdf")
    subprocess.run([sys.executable, str(script), str(html_path), str(pdf_path), "--portrait"],
                   check=True)
    print(f"wrote {pdf_path}")
    return pdf_path


_pdf_made = _render_pdf(_print_html_path)   # PDF: table + one page of details per design
report_pdf_path = _pdf_made.rename(_report_root / f"eu_and_v3_report_{_ts}.pdf") if _pdf_made else None
if _pdf_made:
    _print_html_path.unlink()   # only the PDF's source
print(f"report -> {report_pdf_path or _print_html_path}")

# Part 2 — AND report

A = 5' half, B = 3' half, fused in all ordered pairs of the top `AND_TOP_N` designs of part 1; four states per candidate
(alone, +A, +B, +A+B) and mismatched-trigger controls. Names in this part end in `_and` where they would otherwise clash with part 1.
Do not reassign `scored`, `designs` or `folder` of part 1 here.


## AND candidates: fuse the pairs, then sample the four states of each

A candidate is an ordered pair (A = 5' half, B = 3' half) with a spacer variant. Pairs whose two triggers
are too similar, or whose fused RNA has an AUG before the second hairpin's stem_down (`EXCLUDE_UPSTREAM_AUG`), are excluded before anything is folded; spacers containing an AUG are left out of the pool. The spacer is chosen per pair from a fixed pool as the sequence least
complementary to the toeholds and triggers. Each candidate gets `N_CONTROLS` control states (two mismatched triggers each),
sampled like the other states by default, or taken from the exact matrix as an upper bound with `CONTROLS_SAMPLED = False`.


In [ ]:
import itertools

from _joint_state_parallel import (
    generate_spacer_candidates,
    pick_fake_triggers,
    pick_optimized_spacer,
    trigger_pair_too_similar,
)
from _report_stats import noise_summary, num_text, open_value, spearman, state_ok_and
from _sampled_and_parallel import AndJob, run_parallel_and
from _switch_construct import fuse, upstream_augs

# The pool is the top AND_TOP_N designs of the single report above (its tied group first, then by
# ON - OFF), taken from memory: nothing is scored or sampled again here.
N_SINGLE = N_DESIGNS
_single_csv = all_csv_path   # the table of the single report, named in the AND report's text
pool = []
for _row in scored[:AND_TOP_N]:
    _d = _row["design"]
    _d["single_rank"] = _row["our_rank"]
    _d["single_diff_pp"] = _row["diff_pp"]
    pool.append(_d)
single_results = {}   # the AND results file keeps the key; the singles are in the single report
_saved_and = json.loads(Path(RESULTS_TO_LOAD_AND).read_text()) if RESULTS_TO_LOAD_AND is not None else None
print(f"pool = top {len(pool)} of the single report: "
      f"{[(d['single_rank'], d['rank']) for d in pool]}")


In [ ]:
SPACER_CANDIDATES = generate_spacer_candidates(
    SPACER_CANDIDATE_POOL_SIZE, SPACER_MIN_LENGTH, SPACER_MAX_LENGTH, seed=SPACER_SEED,
)
if EXCLUDE_UPSTREAM_AUG:   # a spacer must not add a start codon upstream of the second hairpin
    SPACER_CANDIDATES = [_s for _s in SPACER_CANDIDATES if "AUG" not in _s]

_control_pool_and = designs + [{"rank": f"extra{_k}", "trigger": _t} for _k, _t in enumerate(CONTROL_EXTRA_TRIGGERS)]
candidates = {}          # key -> dict(d5, d3, built, spacer_label, spacer_seq)
_and_jobs = []
_n_self = _n_similar = _n_upstream_aug = _n_fuse_failed = 0
for _label in SPACER_VARIANTS:
    for _d5, _d3 in itertools.product(pool, pool):
        if _d5["rank"] == _d3["rank"]:
            _n_self += 1
            continue
        if trigger_pair_too_similar(_d5["trigger"], _d3["trigger"],
                                    max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                                    wobble_max_fraction=WOBBLE_MAX_FRACTION):
            _n_similar += 1
            continue
        if _label == "spacer":
            _spacer_seq, _ = pick_optimized_spacer(_d5, _d3, SPACER_CANDIDATES)
        else:
            _spacer_seq = ""
        _built, _problem = fuse(_d5, _d3, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER,
                                spacer=_spacer_seq)
        if _problem is not None:
            _n_fuse_failed += 1
            continue
        if EXCLUDE_UPSTREAM_AUG and upstream_augs(_built):
            _n_upstream_aug += 1
            continue
        if AND_MAX_CANDIDATES is not None and len(candidates) >= AND_MAX_CANDIDATES:
            continue
        _key = f"{_d5['rank']}+{_d3['rank']}|{_label}"
        # controls: N_CONTROLS states, each with two mismatched triggers (real triggers of other designs,
        # at most CONTROL_MAX_OVERLAP_NT nt of identity with either real trigger)
        _real = (_d5["trigger"], _d3["trigger"])
        _fakes = pick_fake_triggers(_real, {_d5["rank"], _d3["rank"]}, _control_pool_and, 2 * N_CONTROLS,
                                    start_offset=len(candidates), max_overlap_nt=CONTROL_MAX_OVERLAP_NT)
        _pairs = tuple((_fakes[_k]["trigger"], _fakes[_k + 1]["trigger"])
                       for _k in range(0, len(_fakes) - 1, 2))
        candidates[_key] = {"d5": _d5, "d3": _d3, "built": _built,
                            "spacer_label": _label, "spacer_seq": _spacer_seq, "controls": _pairs}
        _and_jobs.append(AndJob(
            key=_key, fused=_built["fused"], trigger_a=_d5["trigger"], trigger_b=_d3["trigger"],
            stem_up_a=_built["domains_5p"]["stem_up"], stem_down_a=_built["domains_5p"]["stem_down"],
            stem_up_b=_built["domains_3p"]["stem_up"], stem_down_b=_built["domains_3p"]["stem_down"],
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, seed=SEED,
            controls=candidates[_key]["controls"], sample_controls=CONTROLS_SAMPLED,
        ))
print(f"{len(candidates)} candidates ({_n_self} self-pairs, {_n_similar} excluded as trigger pairs too "
      f"similar, {_n_upstream_aug} with an AUG before the second stem_down, {_n_fuse_failed} failed fuse())")

if RESULTS_TO_LOAD_AND is None:
    _t0_and = time.time()
    and_results = run_parallel_and(_and_jobs, max_workers=MAX_WORKERS)
    and_check = run_parallel_and(
        [j._replace(seed=SEED + CHECK_SEED_OFFSET) for j in _and_jobs], max_workers=MAX_WORKERS)
    print(f"sampled {len(_and_jobs)} candidates x 4 states twice (second run = noise check) in "
          f"{time.time() - _t0_and:.0f}s")
    _results_path_and = _report_root / f"eu_and_v3_and_results_{datetime.datetime.now():%Y%m%dT%H%M%S}.json"
    _results_path_and.write_text(json.dumps(
        {"singles": single_results, "and": and_results, "and_check": and_check}))
    print(f"wrote {_results_path_and}")
else:
    and_results, and_check = _saved_and["and"], _saved_and["and_check"]
    if any(state.get("open_frac_sampled") is None
           for _run in (and_results, and_check) for _res in _run.values()
           for state in (_res[_s] for _s in ("alone", "+A", "+B", "+A+B"))):
        raise ValueError(f"{RESULTS_TO_LOAD_AND} has no per-hairpin Open (open_frac_sampled): "
                         "it was sampled before ON / OFF were normalised per hairpin; run the sampling again")
    if CONTROLS_SAMPLED and any(c.get("open_frac_sampled") is None
           for _run in (and_results, and_check) for _res in _run.values() for c in _res.get("controls", [])):
        raise ValueError(f"{RESULTS_TO_LOAD_AND} has controls without a sampled Open: it was sampled before the "
                         "controls were sampled; run the sampling again")
    print(f"loaded results for {len(and_results)} candidates from {RESULTS_TO_LOAD_AND}")

## Open, checks and ranking

Per state the sampling gives E[r_A], E[r_B] (`run_a_sampled`, `run_b_sampled`), E[min(r_A, r_B)]
(`open_sampled`), E[k_A], E[k_B] and E[min(k_A, k_B)], and the matrix gives each hairpin's E[k] exactly.
A state is *verified* when the sampled E[k_A] and E[k_B] each match the matrix within `VERIFY_SIGMA` standard errors
(+ `VERIFY_FLOOR_NT` nt) of that hairpin, and its Open does not exceed the matrix bound (r <= k always).

ON_min = 100 x E[min(r_A / L_A, r_B / L_B)]_{+A+B}: in each structure each hairpin's longest open run is taken as a fraction of its own stem, and the smaller
of the two counts (both hairpins open at once, each relative to its own length); the mean over the structures is the joint activation. OFF = 100 x max(Open(alone), Open(+A), Open(+B), Control)
with the same Open = E[min(r_A / L_A, r_B / L_B)] for the OFF states, Control = the largest exact bound min(E[k_A] / L_A, E[k_B] / L_B) over the control states with `CONTROLS_SAMPLED = False`, or the largest sampled Open (default).
The nt-based E[min(r_A, r_B)] is kept for the sampler check. The sum-based ON_sum = 100 x E[r_A + r_B] / (L_A + L_B)
is listed for information only. Candidates are ranked by ON_min - OFF (percentage points); candidates within the seed-to-seed noise of the best form a tied group
ordered by dG = G(+A+B) - min(G_alone + G_A + G_B, G_+A + G_B, G_+B + G_A).


In [ ]:
_STATES = ("alone", "+A", "+B", "+A+B")
_CHECKED = _STATES


def _open_value(state, name):
    # nt value of a state used by the sampler check and the nt columns: E[r_A] + E[r_B] for +A+B (its sum bound),
    # E[min(r_A, r_B)] for the others; the ranking uses the per-hairpin fractions (open_frac_sampled) below
    return open_value(state, name)


def _state_ok_and(state, name, l_a, l_b):
    # per-hairpin tolerance tau_h = VERIFY_SIGMA * sd(k_h) / sqrt(N) + VERIFY_FLOOR_NT; a measured sd of 0
    # is kept (only a missing sd falls back to L / 2), see _report_stats.verification_tolerance
    return state_ok_and(state, name, l_a, l_b, sigma=VERIFY_SIGMA, floor_nt=VERIFY_FLOOR_NT)


def _state_strands(c, state):
    fused, ta, tb = c["built"]["fused"], c["d5"]["trigger"], c["d3"]["trigger"]
    return {"alone": fused, "+A": f"{fused}&{ta}", "+B": f"{fused}&{tb}",
            "+A+B": f"{fused}&{tb}&{ta}"}[state]


def _metrics(res, la, lb):
    # ON_min (joint, used for ranking), ON_sum (information), OFF, differences and ratio of one candidate's states
    opens = {s: _open_value(res[s], s) for s in _STATES}
    # per-hairpin Open, a fraction of the own stem: E[min(r_A / L_A, r_B / L_B)] (ON_min and OFF use it)
    fracs = {s: res[s]["open_frac_sampled"] for s in _STATES}
    off_states = {s: fracs[s] for s in ("alone", "+A", "+B")}
    # controls (two mismatched triggers each); the worst of them enters OFF. Default: the exact matrix bound
    # min(E[k_A] / L_A, E[k_B] / L_B) >= E[min(r_A / L_A, r_B / L_B)] (the sampler is unreliable for mismatched
    # triggers). With CONTROLS_SAMPLED: the sampled Open of each control state, checked against the matrix.
    if CONTROLS_SAMPLED:
        ctrl = [c["open_frac_sampled"] for c in res.get("controls", []) if c["open_frac_sampled"] is not None]
        ctrl_ok = sum(_state_ok_and(c, "+A+B", la, lb) for c in res.get("controls", []))
    else:
        ctrl = [min(c["k_a_matrix"] / la, c["k_b_matrix"] / lb) for c in res.get("controls", [])]
        ctrl_ok = None
    ctrl_pct = 100.0 * max(ctrl) if ctrl else None
    if ctrl:
        off_states["control"] = max(ctrl)
    if opens["+A+B"] is None or fracs["+A+B"] is None or None in off_states.values():
        return {"ctrl_pct": ctrl_pct, "n_controls": len(ctrl), "n_controls_ok": ctrl_ok, "open": opens, "on_pct": None, "on_min_pct": None, "off_pct": None, "off_source": None,
                "diff_pp": None, "diff_min_pp": None, "ratio": None, "ratio_floored": False}
    off_source = max(off_states, key=off_states.get)
    off_pct = 100.0 * off_states[off_source]
    on_sum_pct = 100.0 * opens["+A+B"] / (la + lb)
    on_min_pct = 100.0 * fracs["+A+B"]
    on_pct = on_sum_pct   # indicative ON: E[r_A + r_B] / (L_A + L_B); the ranking uses on_min_pct (joint)
    return {"ctrl_pct": ctrl_pct, "n_controls": len(ctrl), "n_controls_ok": ctrl_ok, "open": opens, "on_pct": on_pct, "on_min_pct": on_min_pct, "off_pct": off_pct,
            "off_source": off_source, "diff_pp": on_pct - off_pct, "diff_min_pp": on_min_pct - off_pct,
            "ratio": on_min_pct / max(off_pct, OFF_FLOOR_PCT), "ratio_floored": off_pct < OFF_FLOOR_PCT}


scored_and = []
for _key, _c in candidates.items():
    _res = and_results[_key]
    _b = _c["built"]
    _la = _b["domains_5p"]["stem_up"][1] - _b["domains_5p"]["stem_up"][0]
    _lb = _b["domains_3p"]["stem_up"][1] - _b["domains_3p"]["stem_up"][0]
    _ok = {s: _state_ok_and(_res[s], s, _la, _lb) for s in _STATES}
    _g = {s: folder.partition(_state_strands(_c, s)) for s in _STATES}
    # dG = G(+A+B) - min over the OFF states of [G(state) + G of the triggers that state lacks, folded alone]
    _gta, _gtb = folder.partition(_c["d5"]["trigger"]), folder.partition(_c["d3"]["trigger"])
    _ref = min(_g["alone"] + _gta + _gtb, _g["+A"] + _gtb, _g["+B"] + _gta)
    _m = _metrics(_res, _la, _lb)
    _m_check = _metrics(and_check[_key], _la, _lb)
    scored_and.append({
        "key": _key, **_c, "res": _res, "ok": _ok, **_m,
        "diff_pp_check": _m_check["diff_pp"], "diff_min_pp_check": _m_check["diff_min_pp"],
        "l_a": _la, "l_b": _lb, "l_min": min(_la, _lb),
        "dg": _g["+A+B"] - _ref,
        "n_controls": len(_res.get("controls", [])),
    })

# the ranking quantity is the JOINT difference ON_min - OFF (diff_min_pp); the sum-based ON - OFF is indicative
scored_and.sort(key=lambda r: (r["diff_min_pp"] is None, -(r["diff_min_pp"] or 0.0), r["key"]))
for _pos_and, _r_and in enumerate(scored_and, start=1):
    _r_and["our_rank"] = _pos_and

# rank by the sum-based ON - OFF as well, for comparison (rank_agreement, top_overlap)
_by_sum = sorted(scored_and, key=lambda r: (r["diff_pp"] is None, -(r["diff_pp"] or 0.0), r["key"]))
for _pos_and, _r_and in enumerate(_by_sum, start=1):
    _r_and["rank_sum"] = _pos_and


# None (shown as n/a) for fewer than two candidates, where a correlation is undefined
rank_agreement = spearman([r["our_rank"] for r in scored_and], [r["rank_sum"] for r in scored_and])
top_overlap = len({r["key"] for r in scored_and[:REPORT_TOP_N]} & {r["key"] for r in _by_sum[:REPORT_TOP_N]})

# seed-to-seed noise of ON_min - OFF: the same candidates sampled again with another seed
noise_median_and, noise_p95_and = noise_summary(
    r["diff_min_pp"] - r["diff_min_pp_check"] for r in scored_and
    if r["diff_min_pp"] is not None and r["diff_min_pp_check"] is not None)
# near-ties: candidates within the noise margin of the best are one tied group, ordered by dG
# (no noise estimate: only the best itself; no candidate with a score: no tied group)
_margin_and = 0.0 if noise_p95_and is None else noise_p95_and
_best_and = scored_and[0]["diff_min_pp"] if scored_and else None
_tied_and = [r for r in scored_and if _best_and is not None and r["diff_min_pp"] is not None
             and _best_and - r["diff_min_pp"] <= _margin_and]
n_tied_with_best_and = len(_tied_and)
_rest_and = [r for r in scored_and if r not in _tied_and]
_tied_and.sort(key=lambda r: (r["dg"], r["key"]))
scored_and = _tied_and + _rest_and
for _pos_and, _r_and in enumerate(scored_and, start=1):
    _r_and["our_rank"] = _pos_and
    _r_and["tied"] = _r_and in _tied_and
if not scored_and:
    print("no eligible AND candidates: every pair was excluded (see the counts above)")
print(f"seed-to-seed |change of ON_min - OFF|: median {num_text(noise_median_and)}, "
      f"95th percentile {num_text(noise_p95_and)} pp; {n_tied_with_best_and} candidates within that of rank 1")
print(f"ranking by ON_min - OFF (used) vs by the sum-based ON - OFF: rank correlation {num_text(rank_agreement, 3)}, "
      f"top-{REPORT_TOP_N} overlap {top_overlap}")
_unverified = [(r["our_rank"], s) for r in scored_and for s in _STATES if not r["ok"][s]]
n_ok = {s: sum(r["ok"][s] for r in scored_and) for s in _CHECKED}
print(f"{len(scored_and)} candidates; states verified: {n_ok}")
print("top 5:", [(r["our_rank"], r["key"], num_text(r["diff_min_pp"], 1)) for r in scored_and[:5]])

## Details for each listed candidate

Parts of the fused construct (hairpin A, optional spacer, hairpin B) and both triggers, the 2D picture of
the minimum-free-energy structure for the OFF state (alone) and the ON state (+A+B), and the exact
base-pair-probability matrix of all four states. `_detail_panel_html` builds one panel; it is also used for the interactive-page export.


In [ ]:
from _report_figures import (
    DOMAIN_COLORS_B, SPACER_COLOR,
)

_HIDDEN = ("exp_gene", "opt_exp_gene")


def _fmt_nt_and(x):
    return "n/a" if x is None else f"{x:.2f}"


def _parts(c):
    # (hairpin, name, start, end, colour) in fused order, payload hidden
    b = c["built"]
    rows = []
    for name, (s, e) in b["domains_5p"].items():
        if e > s and name not in _HIDDEN:
            rows.append(("A", name, s, e, DOMAIN_COLORS.get(name, "#C7D0CA")))
    sp0, sp1 = b["spacer_span"]
    if sp1 > sp0:
        rows.append(("", "spacer", sp0, sp1, SPACER_COLOR))
    for name, (s, e) in b["domains_3p"].items():
        if e > s and name not in _HIDDEN:
            rows.append(("B", name, s, e, DOMAIN_COLORS_B.get(name, "#C7D0CA")))
    return sorted(rows, key=lambda t: t[2])


def _construct_html(c):
    fused = c["built"]["fused"]
    parts = _parts(c)
    visible = max(p[3] for p in parts)
    line = "".join(
        f"<span class='dom' style='background:{col}55' title='{h} {n}'>{fused[s:e]}</span>"
        for h, n, s, e, col in parts
    )
    legend = " &ensp; ".join(
        f"<span class='sw' style='background:{col}'></span>{(n + ' (' + h + ')') if h else n}"
        for h, n, s, e, col in parts
    )
    rows = "".join(
        f"<tr><td>{h or '&ndash;'}</td><td>{n}</td><td class='n'>{s + 1}&ndash;{e}</td>"
        f"<td class='n'>{e - s}</td><td class='seq'>{fused[s:e]}</td></tr>"
        for h, n, s, e, col in parts
    )
    rows += (f"<tr><td>A</td><td>trigger A</td><td class='n'>&ndash;</td>"
             f"<td class='n'>{len(c['d5']['trigger'])}</td><td class='seq'>{c['d5']['trigger']}</td></tr>"
             f"<tr><td>B</td><td>trigger B</td><td class='n'>&ndash;</td>"
             f"<td class='n'>{len(c['d3']['trigger'])}</td><td class='seq'>{c['d3']['trigger']}</td></tr>")
    return (f"<p class='note'>Fused switch, 5&prime; to 3&prime; through the Kozak of hairpin B ({visible} nt); "
            f"hairpin A is the 5&prime; half, hairpin B the 3&prime; half:</p>"
            f"<p class='seqline'>{line}</p><p class='note'>{legend}</p>"
            f"<table class='booktabs small'><tr><th>hairpin</th><th>part</th>"
            f"<th class='n'>positions (1-based)</th><th class='n'>length (nt)</th>"
            f"<th>sequence (5&prime; to 3&prime;)</th></tr>{rows}</table>"), visible


def _state_figures_and(c, state, visible, heading):
    strands = _state_strands(c, state)
    fused = c["built"]["fused"]
    sequence = strands.replace("&", "")
    structure = folder.mfe(strands).structure.replace("&", "")
    b = c["built"]
    fills = (
        domain_fill({n: v for n, v in b["domains_5p"].items() if n not in _HIDDEN}, DOMAIN_COLORS, " (A)")
        + domain_fill({n: v for n, v in b["domains_3p"].items() if n not in _HIDDEN}, DOMAIN_COLORS_B, " (B)")
    )
    sp0, sp1 = b["spacer_span"]
    if sp1 > sp0:
        fills.append((sp0, sp1, SPACER_COLOR, "spacer"))
    fills = [f for f in fills if f[0] < visible]
    svg = structure_svg(
        folder.layout_coordinates(structure), sequence, structure,
        f"{heading} (minimum-free-energy structure)", domain_fill=fills,
        visible_len=visible, construct_len=len(fused),
    )
    return svg


def _bpp_png(c, state, dpi=120):
    strands = _state_strands(c, state)
    fused = c["built"]["fused"]
    parts = strands.split("&")
    junctions = [len(fused)] if len(parts) > 1 else []
    if len(parts) > 2:
        junctions.append(len(fused) + len(parts[1]))
    return bpp_heatmap_png(strands.replace("&", ""), folder.base_pair_probabilities(strands),
                           f"{state}: base-pair probabilities", junctions, dpi=dpi)


def _fmt_frac_and(x):
    return "n/a" if x is None else f"{100.0 * x:.1f}%"


def _state_text(c, state):
    r = c["res"][state]
    if state == "+A+B":
        return (f"Open (sum, information) = E[r_A] + E[r_B] = {_fmt_nt_and(r['run_a_sampled'])} + {_fmt_nt_and(r['run_b_sampled'])} = "
                f"{_fmt_nt_and(c['open']['+A+B'])} nt of {c['l_a'] + c['l_b']}; "
                f"Open (min, used for ON) = E[min(r_A / L_A, r_B / L_B)] = {_fmt_frac_and(r['open_frac_sampled'])} "
                f"(in nt, E[min(r_A, r_B)] = {_fmt_nt_and(r['open_sampled'])} with L_A = {c['l_a']}, L_B = {c['l_b']}); "
                f"hairpin A: E[k] {_fmt_nt_and(r['k_a_sampled'])} (matrix {_fmt_nt_and(r['k_a_matrix'])}), "
                f"hairpin B: E[k] {_fmt_nt_and(r['k_b_sampled'])} (matrix {_fmt_nt_and(r['k_b_matrix'])})")
    return (f"Open = E[min(r_A / L_A, r_B / L_B)] = {_fmt_frac_and(r['open_frac_sampled'])}; E[min(k_A, k_B)] = {_fmt_nt_and(r['k_min_sampled'])} nt; "
            f"hairpin A: E[k] {_fmt_nt_and(r['k_a_sampled'])} (matrix {_fmt_nt_and(r['k_a_matrix'])}), "
            f"hairpin B: E[k] {_fmt_nt_and(r['k_b_sampled'])} (matrix {_fmt_nt_and(r['k_b_matrix'])})")


def _detail_panel_html(_r_and, dpi=120):
    _construct, _visible = _construct_html(_r_and)
    _svg_off = _state_figures_and(_r_and, "alone", _visible, "OFF: alone")
    _svg_on = _state_figures_and(_r_and, "+A+B", _visible, "ON: +A+B")
    _pngs = "".join(
        f"<figure><img src='data:image/png;base64,{_bpp_png(_r_and, s, dpi)}'><figcaption>{s}</figcaption></figure>"
        for s in _STATES
    )
    _spacer = f"spacer {len(_r_and['spacer_seq'])} nt" if _r_and["spacer_seq"] else "no spacer"
    return (
        f"<div class='panel' id='p{_r_and['our_rank']}'><h3>Candidate rank {_r_and['our_rank']} &middot; "
        f"A = single rank {_r_and['d5']['single_rank']}, B = single rank {_r_and['d3']['single_rank']} "
        f"&middot; {_spacer} &middot; stems L<sub>A</sub> = {_r_and['l_a']}, L<sub>B</sub> = {_r_and['l_b']} nt "
        f"<a class='back' href='#results'>&uarr; back to the table</a></h3>"
        f"{_construct}"
        f"<div class='state'><h4>OFF: alone</h4><p class='note'>{_state_text(_r_and, 'alone')}</p>{_svg_off}</div>"
        f"<div class='state'><h4>ON: +A+B</h4><p class='note'>{_state_text(_r_and, '+A+B')}</p>{_svg_on}</div>"
        f"<div class='bppRow'>{_pngs}</div>"
        f"<p class='note'>+A: {_state_text(_r_and, '+A')}<br>+B: {_state_text(_r_and, '+B')}</p></div>"
    )


for _r_and in scored_and[:REPORT_TOP_N]:
    _r_and["detail_html"] = _detail_panel_html(_r_and)
print(f"details built for {min(REPORT_TOP_N, len(scored_and))} candidates")

## Report (PDF and CSV of part 2)

Landscape. The E[k] columns and the sampler check are in Appendix A at the end of the report.


In [ ]:
def _fmt_nt_pct_and(x, length):
    return "n/a" if x is None else f"{x:.2f}&nbsp;({100.0 * x / length:.0f}%)"


def _fmt_pp_and(x):
    return "n/a" if x is None else f"{x:.1f}"


def _fmt_ratio_and(r):
    if r["ratio"] is None:
        return "n/a"
    return f"{r['ratio']:.0f}" + ("*" if r["ratio_floored"] else "")


_CSS_and = r'''
@import url("https://cdn.jsdelivr.net/gh/aaaakshat/cm-web-fonts@latest/fonts.css");
@page { size: A4 landscape; margin: 16mm 14mm; }
body { font-family: "Computer Modern Serif", "CMU Serif", "Latin Modern Roman", "Times New Roman", serif;
       font-size: 11.5pt; line-height: 1.5; color: #111; max-width: 262mm; margin: 0 auto; }
h1 { font-size: 20pt; font-weight: normal; text-align: center; margin: 0 0 4pt 0; line-height: 1.25; }
.byline { text-align: center; font-size: 10pt; color: #444; margin-bottom: 18pt; }
h2 { font-size: 13.5pt; margin: 22pt 0 6pt 0; font-weight: bold; break-after: avoid; }
p { margin: 6pt 0; text-align: justify; }
dl { margin: 6pt 0; } dt { break-after: avoid; font-weight: bold; float: left; clear: left; width: 38mm; }
dd { break-inside: avoid; margin: 0 0 4pt 40mm; }
code { font-family: "Courier New", monospace; font-size: 9.5pt; }
table { border-collapse: collapse; margin: 8pt auto; font-size: 6.6pt; }
table.booktabs { border-top: 1.5pt solid #111; border-bottom: 1.5pt solid #111; }
table.booktabs th { border-bottom: 0.75pt solid #111; font-weight: bold; padding: 3pt 3pt; text-align: left; }
table.booktabs th.n { white-space: nowrap; }
table.booktabs td { padding: 2pt 1.5pt; }
td.n, th.n { text-align: right; font-variant-numeric: tabular-nums; }
figure { margin: 10pt 0; text-align: center; break-inside: avoid; }
figcaption { font-size: 10pt; color: #333; margin-top: 4pt; text-align: center; }
.note { font-size: 10pt; color: #333; }
.katex { font-size: 1.0em !important; } .katex-display { margin: 0.55em 0 !important; }
th.grp { text-align: center; border-bottom: 0.75pt solid #111; }
th.on { background: #fbe9e6; } th.off { background: #e6eef7; } th.ctl { background: #eef3e6; }
.detail-link, .back, .detail-btn { white-space: nowrap; font-family: inherit; font-size: 9.5pt; color: #1a5fb4;
  text-decoration: underline; font-weight: normal; border: none; background: none; padding: 0; cursor: pointer; }
.back { margin-left: 14pt; }
.seqline { font-family: "Courier New", monospace; font-size: 9pt; word-break: break-all; line-height: 1.7; }
.dom { padding: 1pt 0; } .sw { display: inline-block; width: 8pt; height: 8pt; margin-right: 3pt; }
tr.detail > td { padding: 8pt 4pt 14pt 4pt; border-bottom: 0.75pt solid #999; }
.panel h3 { font-size: 12pt; margin: 0 0 6pt 0; } .state h4 { font-size: 11pt; margin: 6pt 0 2pt 0; }
.state { margin-top: 10pt; }
.bppRow { display: flex; flex-wrap: wrap; gap: 8pt; justify-content: center; margin-top: 10pt; }
.bppRow figure { flex: 0 0 45%; margin: 2pt; break-inside: avoid; } .bppRow img { max-width: 100%; }
td.seq { font-family: "Courier New", monospace; font-size: 8.5pt; word-break: break-all; }
table.small { font-size: 8.5pt; margin: 4pt 0; }
@media print { .detail, .detail-btn, .noprint { display: none !important; } }
.abstract { border-left: 3pt solid #1F6FEB; padding: 2pt 8pt; margin: 8pt 0 12pt 0; font-size: 11pt; background: #f4f7fb; }
.appendix .panel { break-before: page; } .appendix .panel:first-of-type { break-before: auto; } h2 + table { break-inside: avoid; } table.small, table.small tr { break-inside: avoid; } #results + p { break-after: avoid; break-inside: avoid; } .appendix .state { break-inside: avoid; }
.appendix svg { max-height: 112mm !important; }
figure.construct svg { width: 100%; max-width: 165mm; height: auto; }
figure.construct figcaption { text-align: left; }
.appendix .bppRow { margin-top: 4pt; }
'''

_TEMPLATE_and = r'''<!doctype html><html><head><meta charset="utf-8"><title>eu_and_v3_and report</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.9/dist/contrib/auto-render.min.js"></script>
<script>document.addEventListener("DOMContentLoaded", function () {
  renderMathInElement(document.body, {delimiters: [
    {left: "\\[", right: "\\]", display: true}, {left: "\\(", right: "\\)", display: false}]});
});</script>
<style>@@CSS@@</style></head><body>

<h1>AND screen</h1>
<div class="byline">@@SOURCE@@<br>@@POOL_BYLINE@@ fused in
all ordered pairs &middot; folded at @@TEMPERATURE@@&nbsp;&deg;C, @@SALT@@&nbsp;M salt, with @@ENGINE@@ &middot; @@DATE@@</div>

<div class="abstract"><b>Summary.</b> @@ABSTRACT@@</div>

<h2>1&ensp;The construct and definitions</h2>
<p>An AND candidate fuses two single switches into one RNA (Figure 1). <b>Hairpin A</b> (blue) is the 5&prime; half: everything of the
first design before its own Kozak (prefix, toehold, stem, loop). <b>Hairpin B</b> (magenta) is the 3&prime; half: everything of the
second design after its own prefix (toehold, stem, loop, Kozak and the payload that follows). A <b>spacer</b>
may sit between them (see below); both variants, <i>no_spacer</i> and <i>spacer</i>, are run. The pool is @@POOL_TEXT@@, taken in all
ordered pairs (A, B and B, A are both included, so each pair of hairpins appears twice, with the two orders of the hairpins) without self-pairs and without trigger pairs that are too similar (a run of more than
@@OVERLAP@@ nt of identity, or of complementarity with at most @@WOBBLE_PCT@@% G&middot;U wobble, between trigger A and trigger B)@@AUG_TEXT@@.</p>
<figure class="construct">@@FIG_CONSTRUCT@@<figcaption><b>Figure 1.</b> The fused construct. <i>Top, OFF:</i> the switch alone, both hairpins closed. <i>Bottom, ON (+A+B):</i> each trigger bound to the toehold and \(\mathrm{stem}_{\uparrow}\) of its hairpin; the stems are open and the loops and \(\mathrm{stem}_{\downarrow}\) sides are released. Hairpin A is drawn blue, hairpin B magenta, the spacer teal and the Kozak sequence green; the payload is folded but not drawn.</figcaption></figure>
<p><b>The spacer</b> (teal in Figure 1). A spacer is a short RNA inserted between hairpin A and hairpin B so that the two hairpins do not interfere with each other. It is not designed per pair from scratch: a fixed pool of @@SPACER_POOL@@ random candidate sequences is generated once (seed @@SPACER_SEED@@, so every run gives the same pool), with lengths cycling through @@SPACER_RANGE@@ nt and a bias towards A and U (35% each, 15% each G and C), because A&ndash;U pairs are weaker than G&ndash;C and a low-GC spacer is less likely to pair with anything. For each candidate pair, the candidate whose longest complementary run against the two toeholds and the two triggers is the shortest is chosen (ties: lower GC content, then pool order), i.e. the spacer that is least able to hybridise with them. The spacer is part of the fused RNA, so the numbers of the <i>spacer</i> variant include its effect; the <i>no_spacer</i> variant joins the halves directly.</p>
<dl>
<dt>single rank</dt><dd>@@SINGLE_RANK_TEXT@@ &ldquo;A, B&rdquo; in the table gives the single ranks of the
two designs fused, hairpin A first. Their sequences are listed in each candidate's details.</dd>
<dt>prefix, Kozak, payload</dt><dd>(Figure 1, OFF) a switch RNA starts with a short <i>prefix</i>, then the toehold and the hairpin; the <i>Kozak</i> sequence follows the hairpin and the
<i>payload</i> (the coding sequence) follows the Kozak. In the fused construct only the 5&prime; half's prefix remains (the 3&prime; half starts after its own prefix) and only
the 3&prime; half's payload; the payload is folded but not drawn.</dd>
<dt>how to read the figures</dt><dd>Figure 1 is a schematic; the other figures are computed. 2D structures: every circle is a nucleotide coloured by part (hairpin A and B use different hues); a <b>gold ring</b> marks a nucleotide paired into the payload
(the undrawn &ldquo;hidden tail&rdquo;) and a <b>pink ring</b> one paired to an undrawn trigger strand. Heat maps: the axes run over the whole folded RNA, including the payload and the triggers
present in that state; the dotted red lines mark where one strand ends and the next begins.</dd>
<dt>trigger A, B</dt><dd>(Figure 1, ON) the real triggers of the two designs; trigger A targets hairpin A, trigger B hairpin B.</dd>
<dt>stem</dt><dd>(Figure 1) each hairpin has a stem with two sides of \(L_A\) (resp. \(L_B\)) nucleotides, \(\mathrm{stem}_{\uparrow}\) (5&prime; side) and
\(\mathrm{stem}_{\downarrow}\) (3&prime; side), paired with each other in the closed hairpin. Here \(L_A\), \(L_B\) take the values @@STEM_LENS@@.
Each hairpin's open runs are measured as a fraction of <i>its own</i> stem length (section 2).</dd>
<dt>toehold, hairpin, loop</dt><dd>(Figure 1, OFF) the <i>toehold</i> is the single-stranded stretch where a trigger first binds; a <i>hairpin</i> is a stem (\(\mathrm{stem}_{\uparrow}\) paired with \(\mathrm{stem}_{\downarrow}\)) closed by a <i>loop</i> of unpaired nucleotides. Binding the trigger is meant to open the hairpin.</dd>
<dt>pp</dt><dd>percentage points, the unit of a difference of two percentages.</dd>
<dt>state</dt><dd>the strands folded together: <i>alone</i> (the fused switch, Figure 1 OFF), <i>+A</i>, <i>+B</i>, <i>+A+B</i> (Figure 1 ON; with the named trigger(s)), and @@N_CONTROLS@@ <i>control</i> states (each with two mismatched triggers).</dd>
<dt>mismatched trigger</dt><dd>the real trigger of another design of the first @@N_SINGLE@@ CSV rows (or, if none qualifies, another sequence offered for this purpose) that shares at most @@CONTROL_OVERLAP@@ nt of identical sequence with either real trigger; @@N_CONTROLS@@ pairs of them per candidate are used as wrong inputs in the controls.</dd>
<dt>structure</dt><dd>one folded arrangement \(s\) of the strands, i.e. a set of base pairs.</dd>
<dt>\(E(s)\), \(G\)</dt><dd>the energy of \(s\), and the free energy of the whole ensemble of structures of the state (kcal/mol).</dd>
<dt>\(P(s)\)</dt><dd>the probability of \(s\), \(P(s)=\exp\!\big(-(E(s)-G)/RT\big)\), with \(R\) the gas constant and \(T\) the absolute temperature.</dd>
<dt>open nucleotide</dt><dd>a \(\mathrm{stem}_{\uparrow}\) nucleotide of hairpin \(h\) not paired to a \(\mathrm{stem}_{\downarrow}\) nucleotide of the same hairpin in \(s\) (the open stretches \(r_A\), \(r_B\) of Figure 1, ON).</dd>
<dt>\(k_h(s)\)</dt><dd>the number of open nucleotides of hairpin \(h\in\{A,B\}\) in \(s\), anywhere on its stem.</dd>
<dt>\(r_h(s)\)</dt><dd>the longest stretch of <i>consecutive</i> open nucleotides of hairpin \(h\) in \(s\). Always \(r_h\le k_h\).</dd>
<dt>\(\Delta G\)</dt><dd>\(G_{+A+B}-\min\big(G_{\text{alone}}+G_A+G_B,\;G_{+A}+G_B,\;G_{+B}+G_A\big)\), kcal/mol, from the exact partition function, where \(G_A,G_B\) are the free energies of the triggers folded alone: the free energy gained by forming the two-trigger complex relative to the most stable OFF state with its missing triggers kept separate. More negative: the pair of triggers is favoured more strongly. Orders only the tied group (note that a more stable hairpin makes \(\Delta G\) less negative, so ordering by it slightly favours less stable switches). Strand concentrations are not included (the duplex-initiation term of @@ENGINE_NAME@@'s dimer energy is).</dd>
</dl>

<h2>2&ensp;Open, ON and OFF</h2>
<p><b>ON (+A+B), joint.</b> In a structure \(s\), hairpin A has a longest run \(r_A(s)\) of consecutive open nucleotides and hairpin B a longest run \(r_B(s)\). The AND works when <i>both</i> hairpins are open <i>at once</i>, so the ON used for the ranking looks, in each structure, at the longest open run of each hairpin as a fraction of <i>its own</i> stem and keeps the smaller of the two (both hairpins open together, each relative to its own length):
\[ \mathrm{Open}_{+A+B}^{\min}=\mathbb{E}\Big[\min\Big(\frac{r_A}{L_A},\frac{r_B}{L_B}\Big)\Big]=\sum_{s}P(s)\min\Big(\frac{r_A(s)}{L_A},\frac{r_B(s)}{L_B}\Big),\qquad
\mathrm{ON}_{\min}=100\,\mathrm{Open}_{+A+B}^{\min}. \]
<b>Sum-based ON (indicative).</b> The report also lists the expected sum of the two longest runs, divided by both stem lengths,
\[ \mathrm{ON}_{\Sigma}=100\,\frac{\mathbb{E}[r_A+r_B]}{L_A+L_B}=100\,\sum_{s}P(s)\,\frac{r_A(s)+r_B(s)}{L_A+L_B}, \]
which is 100% when both stems are open over their full length and about 50% when only one is, whether or not the two are open in the same structure. It shows how far the hairpins open overall; it does not order the candidates.</p>
<p><b>OFF.</b> For the states that should stay closed the quantity is the same joint one as for ON, \(\mathrm{Open}=\mathbb{E}[\min(r_A/L_A,r_B/L_B)]\), and OFF is the worst (largest) over the states:
\[ \mathrm{OFF}=100\,\max\big(\mathrm{Open}_{\text{alone}},\mathrm{Open}_{+A},\mathrm{Open}_{+B},\mathrm{Control}\big). \]
(In the +A state hairpin A is meant to open but hairpin B should not, so the minimum is small; likewise for +B. The Control is defined below.) ON and OFF therefore use the same rule, so \(\mathrm{ON}_{\min}-\mathrm{OFF}\) compares like with like.</p>
<p>The candidates are compared with \(\mathrm{ON}_{\min}-\mathrm{OFF}\) (percentage points) and \(\mathrm{ratio}=\mathrm{ON}_{\min}/\max(\mathrm{OFF},@@FLOOR@@)\). <b>They are ranked by \(\mathrm{ON}_{\min}-\mathrm{OFF}\)</b>, largest first, with near-ties handled as described below. Each hairpin is measured relative to its own stem, so a long stem that is only partly open is not hidden behind a short, fully open one; stem lengths are still not fully neutral (fraying at the stem ends is roughly constant in nucleotides, which weighs more on a short stem). The sum-based \(\mathrm{ON}_{\Sigma}-\mathrm{OFF}\) is shown beside it for information and orders nothing, and so is the ratio; the floor of @@FLOOR@@% on OFF keeps the ratio finite, and a ratio computed with the floor is marked with an asterisk.</p>
<p><b>What the ranking represents.</b> It rewards a joint response: both hairpins open in the same structure with both triggers, and no state in which both are open without them. It does not give credit for one hairpin opening alone, which the sum-based ON would score as 50% (one hairpin fully open, the other closed: \(\mathrm{ON}_{\Sigma}=50\%\) but \(\mathrm{ON}_{\min}=0\)). The summary lists how many of the listed candidates have \(\mathrm{ON}_{\min}\) below half of \(\mathrm{ON}_{\Sigma}\); for those the two hairpins respond unevenly.</p>
@@CTRL_PARA@@
<p><b>Ties.</b> Repeating the sampling with another seed changes ON<sub>min</sub> &minus; OFF by a small amount (section 5). Candidates within the 95th percentile of that change from the best one cannot be told apart from it; they form one <i>tied group</i> (&dagger; in the table), ranked inside it by \(\Delta G\) (more negative first) and listed first. Only Monte-Carlo noise is counted; model error is larger, so candidates just outside the group are not clearly worse. The tied group is the shortlist; inside it the order is by \(\Delta G\), so rank 1 need not have the highest ON<sub>min</sub> &minus; OFF. The two orders of one pair (A, B and B, A) and the two spacer variants of a pair are separate rows and often tie.</p>
<p><b>What Open does and does not measure.</b> The +A+B state is the ensemble with both triggers present as a complex; trigger concentrations, binding kinetics and the fraction of constructs that form the complex are not modelled. Open measures how far the hairpin stems are opened at equilibrium (the exact ensemble of @@ENGINE_NAME@@ at the stated temperature). It is a structural proxy
for the hairpins releasing the regions behind them; it is not a measurement of translation, of ribosome binding or start-codon accessibility, of kinetics, or of behaviour in a cell, and trigger accessibility and self-structure, cross-talk between triggers, the openness of the stem_down sides and of the start codon, temperature or parameter-set sensitivity, and the origin of the CSV are not assessed; hairpin A is treated as a second input whose opening is required, its mechanism of gating the payload is assumed, not modelled; the controls are at most @@N_CONTROLS@@ pairs of mismatched triggers per candidate (the number actually found for each candidate is counted in the summary table; with fewer, or none, OFF rests on alone, +A, +B and the controls that exist), and there are no scrambled triggers or known reference construct@@CTRL_LIMIT@@. The pool is preselected (the top of the CSV, then the best singles by the same kind of metric), so the candidates recombine a few hairpins.</p>
<p>Next to each Open the report lists the expected number of open nucleotides anywhere on the stems: \(\mathbb{E}[k_A+k_B]\) for +A+B and \(\mathbb{E}[\min(k_A,k_B)]\) for the
other states (\(\mathrm{Open}\le\mathbb{E}[k]\) in every case). It is the more relevant number for the states that should stay closed: stems open in scattered places have a small
Open but may have a larger \(\mathbb{E}[k]\).</p>

<h2>3&ensp;Estimation and verification</h2>
<p><i>In plain words:</i> Open is estimated by drawing random structures with the probability they have at equilibrium. The exact base-pair probability matrix cannot give Open but can give the average number of open nucleotides, so it is used to check that the drawn structures behave correctly.</p>
<p><b>Sampled estimate.</b> The sum over all structures cannot be written out, so \(N\) structures \(S_1,\dots,S_N\) are drawn from the ensemble
(@@SAMPLER_METHOD@@) and averaged, for example
\[ \mathrm{Open}_{+A+B}^{\min}\approx\frac1N\sum_{i=1}^{N}\min\Big(\frac{r_A(S_i)}{L_A},\frac{r_B(S_i)}{L_B}\Big),\qquad N=@@N_SAMPLES@@ . \]
A structure is drawn with probability \(P(s)\), so this estimates the full sum, including structures that are never drawn. <b>No
normalisation is needed:</b> the draws are already distributed according to \(P(s)\). Its statistical error is \(\mathrm{sd}/\sqrt N\).</p>
@@SAMPLER_HOW@@
<p><b>Matrix check.</b> The base-pair probability matrix \(p_{ij}\) gives each hairpin's expected number of open nucleotides exactly,
\[ \mathbb{E}[k_h]=\sum_{i\in \mathrm{stem}^h_{\uparrow}}\Big(1-\sum_{j\in \mathrm{stem}^h_{\downarrow}}p_{ij}\Big),\qquad h\in\{A,B\}. \]
It cannot give the longest runs, because it does not know which positions are open together in the same structure. It does bound them, since \(r_h\le k_h\):
\(\mathrm{Open}_{+A+B}\le\mathbb{E}[k_A]+\mathbb{E}[k_B]\) and, for the other states, \(\mathrm{Open}\le\min(\mathbb{E}[k_A],\mathbb{E}[k_B])\) (these bounds are written for the runs in nucleotides, which is what the check compares; the per-hairpin Open of section 2 obeys the same bound divided by the stem lengths, \(\min(\mathbb{E}[k_A]/L_A,\mathbb{E}[k_B]/L_B)\), the bound used for the Control). A state is therefore
<i>verified</i> (it is then listed as verified in Appendix A) when (1) the sampled \(\mathbb{E}[k_A]\) and \(\mathbb{E}[k_B]\) each agree with the
matrix within \(\tau_A\), \(\tau_B\), which shows the sampler reproduces the per-nucleotide openness of that state to within \(\tau\) (it does not prove the longest runs themselves), and (2) its Open does not exceed the matrix bound by more than the corresponding \(\tau\); otherwise it is listed as not verified.</p>
<p><b>The tolerance.</b> For each hairpin, \(\tau_h=@@VERIFY_SIGMA@@\,\mathrm{sd}(k_h)/\sqrt N+@@VERIFY_FLOOR@@\) nucleotides, where \(\mathrm{sd}(k_h)\) is the standard deviation of \(k_h\) over the \(N\) draws of that state: a state that should stay closed is checked tightly, a variable one loosely. The constant only allows for the rounding of the matrix. Verification catches gross sampler faults; \(\tau\) can be as large as the seed-to-seed noise of the ranking, so passing the check does not guarantee ranking-level accuracy. The tolerance only decides whether a state counts as verified; it changes no Open or \(\mathbb{E}[k]\) value.</p>
<p><b>Single-input leakage.</b> Because OFF uses the minimum of the two runs, a hairpin that opens on its own while the other stays closed does not raise OFF (it counts only if the other hairpin is open too). This relies on hairpin A actually gating expression, which is assumed here. The per-hairpin E[k] values in the details show such leakage; the junction between the two halves can also change a hairpin's behaviour compared with its single switch.</p>
<p><b>Check for +A+B.</b> For the +A+B state the matrix bounds the sum of the two longest runs, \(\mathbb{E}[r_A+r_B]\le\mathbb{E}[k_A]+\mathbb{E}[k_B]\), and the check compares the sampled sum with this bound (in nucleotides); the ON used for the ranking, \(\mathbb{E}[\min(r_A/L_A,r_B/L_B)]\), is never larger than \(\min(\mathbb{E}[k_A]/L_A,\mathbb{E}[k_B]/L_B)\) by the same argument. The per-hairpin agreement of the sampled and exact \(\mathbb{E}[k]\) is the other part of the check.</p>
<p><b>Unverified states.</b> States that fail the check stay in all medians, noise figures and rank correlations; they are listed in section 5. The sampler's dependence on strand order is empirical, not explained.</p>
@@SAMPLER_LIMIT@@

<h2>4&ensp;Parameters</h2>
<table class="booktabs"><tr><th>name</th><th>value</th><th>meaning</th></tr>
@@PARAMS@@
</table>

<h2>5&ensp;Summary</h2>
<table class="booktabs">
<tr><td>candidates scored</td><td class="n">@@N_SCORED@@</td></tr>
<tr><td>excluded: self-pairs / trigger pairs too similar / an AUG before the second stem<sub>down</sub> (a competing start codon) / fuse failed (the two halves could not be joined into one RNA)</td><td class="n">@@N_EXCL@@</td></tr>
<tr><td>median ON<sub>min</sub> / OFF (% of each hairpin's own stem)</td><td class="n">@@MED_ON@@ / @@MED_OFF@@</td></tr>
<tr><td>median ON<sub>min</sub> &minus; OFF (percentage points)</td><td class="n">@@MED_DIFF@@</td></tr>
<tr><td>for information: median ON<sub>&Sigma;</sub> (% of both stems) / ON<sub>&Sigma;</sub> &minus; OFF (pp)</td><td class="n">@@MED_ON_SUM@@ / @@MED_DIFF_SUM@@</td></tr>
<tr><td>@@CTRL_MED@@, % of each hairpin's own stem (n/a when no candidate has a control)</td><td class="n">@@MED_CTRL@@</td></tr>
<tr><td>candidates with the full @@N_CONTROLS@@ control states / fewer / none (OFF is then the worst of alone, +A, +B and the controls found)</td><td class="n">@@CTRL_COUNTS@@</td></tr>
<tr><td>seed-to-seed change of ON<sub>min</sub> &minus; OFF, second independent run: median / 95th percentile (pp)</td><td class="n">@@NOISE_MED@@ / @@NOISE_P95@@</td></tr>
<tr><td>ranking by ON<sub>min</sub> &minus; OFF (used) vs by ON<sub>&Sigma;</sub> &minus; OFF: rank correlation / overlap of the top @@REPORT_TOP_N@@</td><td class="n">@@RANK_AGREE@@ / @@TOP_OVERLAP@@</td></tr>
<tr><td>listed candidates whose \(\mathrm{ON}_{\min}\) is less than half of \(\mathrm{ON}_{\Sigma}\): one hairpin opens much more than the other</td><td class="n">@@N_JOINT_LOW@@</td></tr>
</table>
<p>@@BEST_TEXT@@</p>

<h2 id="results">6&ensp;First @@REPORT_TOP_N@@ candidates (tied group first, then by \(\mathrm{ON}_{\min}-\mathrm{OFF}\))</h2>
<p class="note">&dagger; = member of the tied group (section 2). * after a ratio = OFF was below the floor. A and B are the single-report ranks of the two designs fused (see the details). \(L\) is the stem length of hairpin A / B.
For +A+B, <i>Open min (ON)</i> is \(\mathbb{E}[\min(r_A/L_A,r_B/L_B)]\), the expected longest run open in both hairpins at once, each as a fraction of its own stem, shown as a percentage; it is the ON used for the ranking. <i>Open sum</i> is \(\mathbb{E}[r_A+r_B]\), the expected sum of the longest open runs of the two stems (percentage of \(L_A+L_B\) in brackets), listed for information only. The expected number of open nucleotides of both stems, \(\mathbb{E}[k_A+k_B]\), sampled and exact (matrix), is in Appendix A at the end.
The ranking is by \(\mathrm{ON}_{\min}-\mathrm{OFF}\) (joint ON, OFF as defined in section 2). @@CTRL_DEF@@ The sampled \(\mathbb{E}[\min(k_A,k_B)]\) of the OFF states and the sampler verification are in <a href="#appendixA">Appendix A</a>; the per-hairpin matrix values are in the details. OFF is the largest of Open(alone), Open(+A), Open(+B) and the Control.</p>
<table class="booktabs">
<thead><tr><th rowspan="2" class="n">rank</th><th rowspan="2" style="white-space:nowrap">single ranks<br>A,&nbsp;B</th><th rowspan="2">spacer</th><th rowspan="2" class="n">\(L_A/L_B\)</th>
<th colspan="2" class="grp on">ON &ensp;<small>+A+B</small></th>
<th colspan="3" class="grp off">OFF &ensp;<small>alone, +A, +B</small></th>
<th class="grp ctl">Control</th>
<th rowspan="2" class="n">ON<sub>min</sub> &minus; OFF<br>(pp, ranking)</th><th rowspan="2" class="n">ON<sub>&Sigma;</sub> &minus; OFF<br>(pp, info)</th><th rowspan="2" class="n">ratio<br>ON<sub>min</sub> / OFF</th><th rowspan="2" class="n">\(\Delta G\)<br>(kcal/mol)<br>lower = tighter</th>@@BTNHEAD@@</tr>
<tr><th class="n">\(\mathrm{Open}_{+A+B}\)<br>sum (info)</th><th class="n">\(\mathrm{Open}_{+A+B}\)<br>min (ON)</th>
<th class="n">\(\mathrm{Open}_{\text{alone}}\)</th><th class="n">\(\mathrm{Open}_{+A}\)</th><th class="n">\(\mathrm{Open}_{+B}\)</th>
<th class="n">@@CTRL_HDR@@</th></tr></thead>
@@ROWS@@
</table>
@@APPENDIX@@
@@APPENDIX2@@
</body></html>'''


def _row_html_and(r):
    res = r["res"]
    button = f"<td><a class='detail-link' href='#p{r['our_rank']}'>Details</a></td>"
    spacer = f"spacer ({len(r['spacer_seq'])}&nbsp;nt)" if r["spacer_seq"] else "no_spacer"
    cells = (
        f"<td class='n'>{r['our_rank']}{'&dagger;' if r['tied'] else ''}</td>"
        f"<td style='white-space:nowrap'>{r['d5']['single_rank']},&nbsp;{r['d3']['single_rank']}</td><td>{spacer}</td>"
        f"<td class='n'>{r['l_a']}/{r['l_b']}</td>"
        f"<td class='n'>{_fmt_nt_pct_and(r['open']['+A+B'], r['l_a'] + r['l_b'])}</td>"
        f"<td class='n'>{_fmt_frac_and(res['+A+B']['open_frac_sampled'])}</td>"
    )
    for s in ("alone", "+A", "+B"):
        cells += f"<td class='n'>{_fmt_frac_and(res[s]['open_frac_sampled'])}</td>"
    cells += (f"<td class='n'>{_fmt_frac_and(None if r['ctrl_pct'] is None else r['ctrl_pct'] / 100.0)}</td>")
    cells += (f"<td class='n'><b>{_fmt_pp_and(r['diff_min_pp'])}</b></td><td class='n'>{_fmt_pp_and(r['diff_pp'])}</td>"

              f"<td class='n'>{_fmt_ratio_and(r)}</td>"
              f"<td class='n'>{r['dg']:.1f}</td>")
    return f"<tr>{cells}{button}</tr>"


_stem_text_and = ", ".join(str(n) for n in sorted({x for r in scored_and for x in (r["l_a"], r["l_b"])}))
_param_rows_and = [
    ("SPACER_CANDIDATE_POOL_SIZE", SPACER_CANDIDATE_POOL_SIZE, "random candidate spacers generated once (A/U-biased, seeded)"),
    ("SPACER_LENGTHS", f"{SPACER_MIN_LENGTH}&ndash;{SPACER_MAX_LENGTH}", "spacer lengths (nt), cycled through the pool"),
    ("SPACER_SEED", SPACER_SEED, "seed of the spacer pool"),
    ("N_SINGLE", N_SINGLE, "first CSV rows scored as single switches"),
    ("AND_TOP_N", AND_TOP_N, "top designs of the single report (or, without it, top singles by ON &minus; OFF) fused in all ordered pairs"),
    ("SPACER_VARIANTS", ", ".join(SPACER_VARIANTS), "both variants are run and ranked together"),
    ("FAKE_MAX_SEQ_OVERLAP_NT", FAKE_MAX_SEQ_OVERLAP_NT,
     "trigger A and trigger B may share at most this many nucleotides in a row (identity or complementarity)"),
    ("WOBBLE_MAX_FRACTION", WOBBLE_MAX_FRACTION, "max share of G&middot;U wobble in a counted complementary run"),
    ("EXCLUDE_UPSTREAM_AUG", EXCLUDE_UPSTREAM_AUG, "exclude constructs with an AUG before the second hairpin's stem_down (and spacers containing AUG)"),
    ("TEMPERATURE_C", TEMPERATURE_C, "folding temperature, &deg;C"),
    ("SALT_MOLAR", SALT_MOLAR, "monovalent salt, mol/L (no magnesium)"),
    ("N_SAMPLES", N_SAMPLES, "structures drawn from the ensemble for each state"),
    ("SEED", SEED, "random seed for the sampler (a rerun repeats)"),
    ("CHECK_SEED_OFFSET", CHECK_SEED_OFFSET, "a second sampling run uses SEED + this; it only measures the noise (section 5)"),
    ("VERIFY_SIGMA", VERIFY_SIGMA, "&tau;: allowed disagreement with the matrix, in standard errors of the sampled mean"),
    ("VERIFY_FLOOR_NT", VERIFY_FLOOR_NT, "added to &tau; (nt)"),
    ("N_CONTROLS", N_CONTROLS, "mismatched control states per candidate (two mismatched triggers each); OFF is the worst of alone, +A, +B and these"),
    ("CONTROL_MAX_OVERLAP_NT", CONTROL_MAX_OVERLAP_NT, "each mismatched control trigger shares at most this many nt of identity with either real trigger"),
    ("OFF_FLOOR_PCT", OFF_FLOOR_PCT, "OFF is not allowed below this many % of the stem in the ratio ON / OFF"),
    ("REPORT_TOP_N", REPORT_TOP_N, "candidates listed in the table of section 6"),
]
_params_html_and = "".join(
    f"<tr><td><code>{n}</code></td><td>{v}</td><td>{m}</td></tr>" for n, v, m in _PROVENANCE_ROWS + _param_rows_and
)
_valid = [r for r in scored_and if r["diff_min_pp"] is not None]
_n_and = len(scored_and)
_NO_CANDIDATES_TEXT = (
    f"<b>No eligible AND candidates.</b> All {_n_self + _n_similar + _n_upstream_aug + _n_fuse_failed} pair variants of the "
    f"{len(pool)} designs of the pool were excluded before folding: {_n_self} self-pairs, {_n_similar} pairs with triggers that are too similar, "
    f"{_n_upstream_aug} constructs with an AUG before the second hairpin's stem<sub>down</sub> and {_n_fuse_failed} that could not be fused. "
    "Nothing was sampled, so there are no scores, no ranking and no correlation. Loosen a screen (<code>FAKE_MAX_SEQ_OVERLAP_NT</code>, "
    "<code>WOBBLE_MAX_FRACTION</code>, <code>EXCLUDE_UPSTREAM_AUG</code>), or use a larger or different pool (<code>AND_TOP_N</code>).")
_fields_and = {
    **_WORDS, **_WORDS_AND,
    "CSS": _CSS_and, "SOURCE": html.escape(Path(CSV_PATH).name),
    "ABSTRACT": (_NO_CANDIDATES_TEXT if not scored_and else f"{_n_and} AND constructs (two single switches fused, A at the 5&prime; end, B at the 3&prime; end) are folded alone, with trigger A, with trigger B, with both and with mismatched triggers (controls). "
        f"<b>ON</b> (ON<sub>min</sub>) is the expected longest run open in both hairpins at once with both triggers, each hairpin relative to its own stem; <b>OFF</b> is the same quantity for the worst of alone, +A, +B and {N_CONTROLS} mismatched-trigger controls (100% = fully open). A good AND has ON near 100% and OFF near 0%. The sum-based ON<sub>&Sigma;</sub> (both hairpins counted separately, as a percentage of both stems) is listed for information. "
        f"Median ON is {median_text((r['on_min_pct'] for r in _valid), unit='%')} and median OFF {median_text((r['off_pct'] for r in _valid), unit='%')}. "
        f"{n_tied_with_best_and} of {_n_and} candidates lie within the run-to-run difference (95th percentile of the change of ON<sub>min</sub> &minus; OFF between two independent samplings: {num_text(noise_p95_and)} points) of the best. "
        f"They are marked &dagger; and are the shortlist (ordered by \\(\\Delta G\\)); any of them is as good as the others by this score. Ranking by the sum-based ON<sub>&Sigma;</sub> &minus; OFF instead would give a different order (Spearman rank correlation {num_text(rank_agreement)}; the overlap of the top {REPORT_TOP_N} in section 5 is the more informative figure). Structure-based prediction only, no experimental calibration." + _SAMPLER_BANNER), "N_SINGLE": N_SINGLE, "AND_TOP_N": AND_TOP_N,
    "POOL_BYLINE": (f"Top {AND_TOP_N} designs of the single report ({html.escape(Path(_single_csv).name)})" if _single_csv is not None
                    else f"First {N_SINGLE} designs of the CSV, top {AND_TOP_N} by single-switch ON &minus; OFF"),
    "POOL_TEXT": (f"the {AND_TOP_N} best designs of the single report <code>{html.escape(Path(_single_csv).name)}</code> (its tied group first, then by ON &minus; OFF; the same ranking as that report)" if _single_csv is not None
                  else f"the {AND_TOP_N} best single switches by ON &minus; OFF, scored here"),
    "SINGLE_RANK_TEXT": ("the rank of a design in the single report (the pool is its top designs);" if _single_csv is not None
                         else "the position of a design when the first rows are ordered by ON &minus; OFF alone (without the tie-group ordering of the single report, so it can differ from that report's rank);"),
    "TEMPERATURE": f"{TEMPERATURE_C:g}", "DATE": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "SPACER_RANGE": f"{SPACER_MIN_LENGTH}&ndash;{SPACER_MAX_LENGTH}", "OVERLAP": FAKE_MAX_SEQ_OVERLAP_NT, "SPACER_POOL": SPACER_CANDIDATE_POOL_SIZE, "SPACER_SEED": SPACER_SEED,
    "AUG_TEXT": (", and without constructs that have an AUG anywhere before the start of the second hairpin's \\(\\mathrm{stem}_{\\downarrow}\\) "
                 "(in the first hairpin, the spacer or the second hairpin's toehold, stem or loop), which would be a start codon competing with the payload's; "
                 "the spacer pool is filtered the same way" if EXCLUDE_UPSTREAM_AUG else ""),
    "WOBBLE_PCT": f"{100 * WOBBLE_MAX_FRACTION:.0f}", "STEM_LENS": _stem_text_and, "FLOOR": f"{OFF_FLOOR_PCT:g}", "CONTROL_OVERLAP": CONTROL_MAX_OVERLAP_NT,
    "N_SAMPLES": N_SAMPLES, "VERIFY_SIGMA": f"{VERIFY_SIGMA:g}", "VERIFY_FLOOR": f"{VERIFY_FLOOR_NT:g}",
    "PARAMS": _params_html_and, "N_SCORED": _n_and,
    "N_EXCL": f"{_n_self} / {_n_similar} / {_n_upstream_aug} / {_n_fuse_failed}",
    "N_CONTROLS": N_CONTROLS,
    "MED_CTRL": median_text((r['ctrl_pct'] for r in _valid), unit='%'),
    "CTRL_COUNTS": (f"{sum(r['n_controls'] >= N_CONTROLS for r in _valid)} / "
                    f"{sum(0 < r['n_controls'] < N_CONTROLS for r in _valid)} / {sum(r['n_controls'] == 0 for r in _valid)}"),
    "MED_ON": median_text((r['on_min_pct'] for r in _valid), unit='%'),
    "MED_ON_SUM": median_text((r['on_pct'] for r in _valid), unit='%'),
    "MED_DIFF_SUM": median_text(r['diff_pp'] for r in _valid),
    "MED_OFF": median_text((r['off_pct'] for r in _valid), unit='%'),
    "MED_DIFF": median_text(r['diff_min_pp'] for r in _valid),
    "REPORT_TOP_N": REPORT_TOP_N,
    "NOISE_MED": num_text(noise_median_and), "NOISE_P95": num_text(noise_p95_and),
    "N_JOINT_LOW": (f"{sum(r['on_min_pct'] is not None and r['on_min_pct'] < 0.5 * r['on_pct'] for r in scored_and[:REPORT_TOP_N])}"
                    f" of {len(scored_and[:REPORT_TOP_N])}"),
    "RANK_AGREE": num_text(rank_agreement, 3), "TOP_OVERLAP": f"{top_overlap} of {REPORT_TOP_N}",
    "BEST_TEXT": ("There is no candidate to rank." if not scored_and else f"Highest ON<sub>min</sub> &minus; OFF: {num_text(max((r['diff_min_pp'] for r in scored_and if r['diff_min_pp'] is not None), default=None), 1)} points. "
                  f"{n_tied_with_best_and} of the {len(scored_and)} candidates lie within the 95th-percentile seed-to-seed change "
                  f"({num_text(noise_p95_and)} points) of it. They form the tied group (&dagger;), the shortlist (these are {len({(r['d5']['single_rank'], r['d3']['single_rank']) for r in scored_and if r['tied']})} ordered pairs of hairpins, {len({frozenset((r['d5']['single_rank'], r['d3']['single_rank'])) for r in scored_and if r['tied']})} unordered, each with up to two spacer variants, so they are far fewer than {n_tied_with_best_and} independent designs); the table lists only the first {REPORT_TOP_N} of them, the others carry tied_group = True in the CSV; inside it candidates are ordered by \\(\\Delta G\\), not by ON<sub>min</sub> &minus; OFF. "
                  f"A state that is not verified (listed in Appendix A) has an unreliable ON or OFF."),
}


def _fill_and(template, fields):
    for _key, _value in fields.items():
        template = template.replace(f"@@{_key}@@", str(_value))
    return template


_top_and = scored_and[:REPORT_TOP_N]


def _appx_row(r):
    res = r["res"]["+A+B"]
    flags = " ".join(("&#10003;" if r["ok"][st_] else "<b>&times;</b>") for st_ in _STATES)
    return (f"<tr><td class='n'>{r['our_rank']}</td>"
            f"<td style='white-space:nowrap'>{r['d5']['single_rank']},&nbsp;{r['d3']['single_rank']}</td>"
            f"<td class='n'>{r['l_a']}/{r['l_b']}</td>"
            f"<td class='n'>{_fmt_nt_and(res['k_a_sampled'] + res['k_b_sampled'])}</td>"
            f"<td class='n'>{_fmt_nt_and(res['k_a_matrix'] + res['k_b_matrix'])}</td>"
            f"<td class='n'>{_fmt_nt_and(r['open']['+A+B'])}</td>"
            + "".join(f"<td class='n'>{_fmt_nt_and(r['res'][st_]['k_min_sampled'])}</td>"
                      for st_ in ("alone", "+A", "+B"))
            + f"<td>{flags}</td>" + (f"<td>{r['n_controls_ok']} of {r['n_controls']}</td>" if CONTROLS_SAMPLED else "") + "</tr>")


_appendix2 = ("<div class='appendix2'><h2 id='appendixA'>Appendix A&ensp;Expected number of open nucleotides and sampler check</h2>"
              "<p class='note'>For the +A+B state: \\(\\mathbb{E}[k_A+k_B]\\) is the expected number of open nucleotides "
              "anywhere on the two stems (not necessarily consecutive), sampled and from the exact matrix; Open sum is the (indicative) sum-based ON, "
              "\\(\\mathbb{E}[r_A+r_B]\\) (it cannot exceed \\(\\mathbb{E}[k_A+k_B]\\)). For the OFF states (alone, +A, +B) the sampled \\(\\mathbb{E}[\\min(k_A,k_B)]\\), which bounds their Open, is listed.</p>"
              f"<p class='note'><b>Sampler check.</b> A state is verified when its sampled \\(\\mathbb{{E}}[k]\\) agrees with the matrix within the tolerance of section 3. In this run "
              f"{' / '.join(str(n_ok[s]) for s in _STATES)} of {_n_and} states passed (alone / +A / +B / +A+B). States not verified (candidate rank, state): "
              f"{', '.join(f'rank {rk}: {st}' for rk, st in _unverified) or 'none'}. The last column of the table shows the four states of the listed candidates in this order (&#10003; = verified, <b>&times;</b> = not). "
              + (f"The control states (two mismatched triggers) are sampled and checked the same way: {sum(r['n_controls_ok'] for r in scored_and)} of "
                 f"{sum(r['n_controls'] for r in scored_and)} passed; the last column gives the count per listed candidate." if CONTROLS_SAMPLED else
                 "The control states are not sampled: each is taken from the exact matrix (section 2), so there is nothing to check.")
              + "</p>"
              "<table class='booktabs'><thead><tr><th class='n'>rank</th><th>single ranks<br>A,&nbsp;B</th><th class='n'>\\(L_A/L_B\\)</th>"
              "<th class='n'>\\(\\mathbb{E}[k_A+k_B]\\)<br>sampled</th><th class='n'>\\(\\mathbb{E}[k_A+k_B]\\)<br>matrix</th>"
              "<th class='n'>Open sum<br>\\(\\mathbb{E}[r_A+r_B]\\)</th>"
              "<th class='n'>alone<br>\\(\\mathbb{E}[\\min(k_A,k_B)]\\)</th><th class='n'>+A<br>\\(\\mathbb{E}[\\min(k_A,k_B)]\\)</th><th class='n'>+B<br>\\(\\mathbb{E}[\\min(k_A,k_B)]\\)</th>"
              "<th>states verified</th>" + ("<th>control states verified</th>" if CONTROLS_SAMPLED else "") + "</tr></thead>"
              + "".join(_appx_row(r) for r in _top_and) + "</table></div>")
_appendix_and = ("<div class='appendix'><h2>7&ensp;Candidate details</h2><p class='note'>Five pages per candidate "
             "of the table above: its parts, the 2D structure of the OFF state (alone), the 2D structure of the "
             "ON state (+A+B), and the base-pair probabilities of all four states.</p>" + "".join(r["detail_html"] for r in _top_and) + "</div>")
report_print_html_and = _fill_and(_TEMPLATE_and, {
    **_fields_and, "BTNHEAD": '<th rowspan="2"></th>', "APPENDIX": _appendix_and, "APPENDIX2": _appendix2,
    "ROWS": "".join(_row_html_and(r) for r in _top_and),
})

_ts_and = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
_print_html_path_and = _report_root / f"eu_and_v3_and_report_{_ts_and}.html"   # the PDF's source
_print_html_path_and.write_text(report_print_html_and)

import csv as _csv
all_csv_path_and = _report_root / f"eu_and_v3_and_all_candidates_{_ts_and}.csv"
with all_csv_path_and.open("w", newline="") as _fh_and:
    _w = _csv.writer(_fh_and)
    _w.writerow(["our_rank", "csv_rank_A", "csv_rank_B", "spacer", "L_A", "L_B", "Open_min_alone_nt",
                 "Open_min_+A_nt", "Open_min_+B_nt", "Open_sum_+A+B_nt", "E_k_sum_+A+B_nt", "ONsum_pct", "ONmin_pct", "OFF_pct",
                 "OFF_source", "ONsum_minus_OFF_pp", "ONmin_minus_OFF_pp", "ratio_ONmin_over_OFF", "dG_kcal_per_mol",
                 "verified_alone", "verified_+A", "verified_+B", "verified_+A+B", "tied_group",
                 _CTRL_CSV, "n_controls", "n_controls_verified", "trigger_len_A", "trigger_len_B", "spacer_len"])
    for r in scored_and:
        _w.writerow([r["our_rank"], r["d5"]["rank"], r["d3"]["rank"], r["spacer_label"], r["l_a"],
                     r["l_b"], r["open"]["alone"], r["open"]["+A"], r["open"]["+B"], r["open"]["+A+B"],
                     r["res"]["+A+B"]["k_a_sampled"] + r["res"]["+A+B"]["k_b_sampled"], r["on_pct"], r["on_min_pct"], r["off_pct"], r["off_source"],
                     r["diff_pp"], r["diff_min_pp"], r["ratio"], r["dg"], *(r["ok"][s] for s in _STATES), r["tied"],
                     r["ctrl_pct"], r["n_controls"], r["n_controls_ok"],
                     len(r["d5"]["trigger"]), len(r["d3"]["trigger"]), len(r["spacer_seq"])])
print(f"wrote {all_csv_path_and}")


def _render_pdf_and(html_path):
    if not MAKE_PDF:
        print("MAKE_PDF is False -- the HTML source of the PDF is kept instead")
        return None
    for base in (Path.cwd(), *Path.cwd().parents):
        script = base / "src/engine/gates/notebooks/toehold/render_pdf.py"
        if script.exists():
            break
    else:
        print("render_pdf.py not found -- HTML written, PDF skipped")
        return None
    pdf_path = html_path.with_suffix(".pdf")
    subprocess.run([sys.executable, str(script), str(html_path), str(pdf_path)], check=True)  # landscape
    print(f"wrote {pdf_path}")
    return pdf_path


_pdf_made_and = _render_pdf_and(_print_html_path_and)   # PDF: table + the details of every candidate
report_pdf_path_and = _pdf_made_and
if _pdf_made_and:
    _print_html_path_and.unlink()   # only the PDF's source
print(f"report -> {report_pdf_path_and or _print_html_path_and}")

## Data for an interactive page

Writes the rows of all candidates (`rows.json`) and the detail panels of the best `ARTIFACT_DETAIL_N` candidates (small scripts in `d/`) next to the report, for a page with filters and expandable structures and heat maps (the page template itself is not in the repository; see the spec, section Outputs).

In [ ]:
def _pct(x):
    return None if x is None else 100.0 * x


if EXPORT_ARTIFACT:
    _art_dir = _report_root / f"eu_and_v3_and_artifact_{_ts_and}"
    (_art_dir / "d").mkdir(parents=True, exist_ok=True)
    _rows = []
    for _r_and in scored_and:
        _res = _r_and["res"]
        _rows.append({
            "rank": _r_and["our_rank"], "tied": bool(_r_and["tied"]),
            "A": _r_and["d5"]["single_rank"], "B": _r_and["d3"]["single_rank"],
            "csvA": int(_r_and["d5"]["rank"]), "csvB": int(_r_and["d3"]["rank"]),
            "spacer": _r_and["spacer_label"], "spacerLen": len(_r_and["spacer_seq"]),
            "LA": _r_and["l_a"], "LB": _r_and["l_b"],
            "trigLenA": len(_r_and["d5"]["trigger"]), "trigLenB": len(_r_and["d3"]["trigger"]),
            "on": _r_and["on_min_pct"], "onSum": _r_and["on_pct"], "off": _r_and["off_pct"], "offSource": _r_and["off_source"],
            "diff": _r_and["diff_min_pp"], "diffSum": _r_and["diff_pp"], "ratio": _r_and["ratio"], "dG": _r_and["dg"],
            "ctrl": _r_and["ctrl_pct"],
            "openAlone": _pct(_res["alone"]["open_frac_sampled"]), "openA": _pct(_res["+A"]["open_frac_sampled"]),
            "openB": _pct(_res["+B"]["open_frac_sampled"]),
            "ok": [bool(_r_and["ok"][_s]) for _s in _STATES],
            "detail": _r_and["our_rank"] <= ARTIFACT_DETAIL_N,
        })
    (_art_dir / "rows.json").write_text(json.dumps(_rows))
    _shard = 4
    for _i in range(0, min(ARTIFACT_DETAIL_N, len(scored_and)), _shard):
        _chunk = {}
        for _r_and in scored_and[_i:_i + _shard]:
            _chunk[_r_and["our_rank"]] = _detail_panel_html(_r_and, dpi=75).replace(
                "<a class='back' href='#results'>&uarr; back to the table</a>", "")
        (_art_dir / "d" / f"s{_i // _shard}.js").write_text(
            "window.DET=window.DET||{};Object.assign(window.DET," + json.dumps(_chunk) + ");")
    _size = sum(f.stat().st_size for f in _art_dir.rglob("*") if f.is_file())
    print(f"artifact data: {len(_rows)} rows, details for {min(ARTIFACT_DETAIL_N, len(scored_and))} candidates, "
          f"{_size / 1e6:.1f} MB in {_art_dir}")


## Trigger positions of the listed AND candidates

For each of the first `REPORT_TOP_N` candidates of the AND report, where trigger A and trigger B sit on the target
transcript (`trigger_start_pos` and `trigger_end_pos` of the CSV, 0-indexed, first nt of the trigger), the gap between them
and the rank of each design in the single report. Blue = trigger A, orange = trigger B. Saved as
`eu_and_v3_and_trigger_positions_<time>.png` next to the AND report; it is not part of the report.

In [ ]:
def _transcript_name_and(csv_path):
    parts = Path(csv_path).parts
    return parts[parts.index("targets") + 1].upper() if "targets" in parts else "the transcript"


def _plot_trigger_positions_and(rows, transcript, out_path):
    blue, orange, ink, muted = "#2b6cb8", "#d9822b", "#1c2530", "#5d6b7a"
    fig, ax = plt.subplots(figsize=(20, 0.65 * len(rows) + 1.5), dpi=105)
    x_max = max(max(r["d5"]["trigger_end"], r["d3"]["trigger_end"]) for r in rows)
    for i, r in enumerate(rows):
        y = len(rows) - 1 - i
        ends = sorted(((d["trigger_start"], d["trigger_end"], side, d, colour)
                       for side, d, colour in (("A", r["d5"], blue), ("B", r["d3"], orange))),
                      key=lambda e: e[0])
        ax.plot([ends[0][1], ends[1][0]], [y, y], color="#a9a9b8", lw=1.2, zorder=1)
        ax.text((ends[0][1] + ends[1][0]) / 2, y + 0.17, f"gap {ends[1][0] - ends[0][1]} nt",
                ha="center", va="bottom", fontsize=8, color=muted)
        for k, (start, end, side, d, colour) in enumerate(ends):
            ax.add_patch(plt.Rectangle((start, y - 0.18), max(end - start, 12), 0.36, color=colour, zorder=2))
            label = f"{side}  single #{d['single_rank']}   start {start}, {end - start} nt"
            if k == 0:
                ax.text(start - 14, y, label, ha="right", va="center", fontsize=9.5, color=colour)
            else:
                ax.text(max(end, start + 12) + 14, y, label, ha="left", va="center", fontsize=9.5, color=colour)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([f"AND #{r['our_rank']}  ({'spacer' if r['spacer_seq'] else 'no spacer'})"
                        for r in rows][::-1], fontsize=10.5)
    ax.set_ylim(-0.7, len(rows) - 0.2)
    ax.set_xlim(-520, x_max + 520)
    ax.set_xticks(range(0, int(x_max) + 500, 500))
    ax.grid(axis="x", color="#e3e7ea", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_xlabel(f"position on the {transcript} transcript (nt, trigger_start_pos from the CSV)", fontsize=11, color=ink)
    ax.set_title(f"Triggers of the first {len(rows)} AND candidates on {transcript} (ranked as in the AND report)\n"
                 "blue = trigger A, orange = trigger B; label = rank in the single report, first nt and length",
                 fontsize=12.5, color=ink)
    fig.tight_layout()
    fig.savefig(out_path, facecolor="white")
    plt.close(fig)


_positions_path_and = _report_root / f"eu_and_v3_and_trigger_positions_{_ts_and}.png"
if scored_and:
    _plot_trigger_positions_and(scored_and[:REPORT_TOP_N], _transcript_name_and(CSV_PATH), _positions_path_and)
    print(f"wrote {_positions_path_and}")
else:
    print("no eligible AND candidates: no trigger-position plot")